# Patterns using gSpan

In [7]:
!pip install gspan-mining

In [8]:
import os
import re
import json
import math
import random
import shutil
import subprocess
from pathlib import Path
from collections import Counter, defaultdict
import glob
import networkx as nx

# Data processing and numerical analysis
import pandas as pd
import numpy as np

# Graph processing
import networkx as nx

# gSpan
from gspan_mining.config import parser
from gspan_mining.main import main
CLASSIFICATION_CSV_URL = (
    "https://raw.githubusercontent.com/disel-espol/"
    "hpc-and-edge-cloud-architectures/main/"
    "Journal_paper/classified_architectures_with_metadata.csv"
)

!git clone https://github.com/WiscADSL/Cloudscape.git

!wget -q -O /content/classified_architectures_with_metadata.csv "$CLASSIFICATION_CSV_URL"

CLASSIFICATION_CSV = "/content/classified_architectures_with_metadata.csv"

df = pd.read_csv("/content/classified_architectures_with_metadata.csv")

fatal: destination path 'Cloudscape' already exists and is not an empty directory.


In [9]:
GRAPH_DIR = "/content/Cloudscape/data/graphs"

files = sorted(glob.glob(os.path.join(GRAPH_DIR, "*.graphml")))

if not files:
    raise FileNotFoundError(f"No se encontraron archivos en {GRAPH_DIR}")

# Revisamos el primer grafo
file_path = files[0]
G = nx.read_graphml(file_path)

print("Archivo:", os.path.basename(file_path))
print("Tipo de grafo:", type(G).__name__)
print("¿Es dirigido?:", G.is_directed())
print("Número de nodos:", G.number_of_nodes())
print("Número de aristas:", G.number_of_edges())

print("\n--- PRIMEROS 10 NODOS ---")
for node_id, attrs in list(G.nodes(data=True))[:10]:
    print("ID:", node_id)
    print("Atributos:", attrs)
    print()

print("\n--- PRIMERAS 10 ARISTAS ---")
for edge in list(G.edges(data=True))[:10]:
    print(edge)

Archivo: -3lnf5lzsH0.graphml
Tipo de grafo: DiGraph
¿Es dirigido?: True
Número de nodos: 13
Número de aristas: 16

--- PRIMEROS 10 NODOS ---
ID: 0
Atributos: {'name': '', 'service': 'OnPremDC', 'notes': ''}

ID: 1
Atributos: {'name': '', 'service': 'EMR', 'notes': ''}

ID: 2
Atributos: {'name': '', 'service': 'Athena', 'notes': ''}

ID: 3
Atributos: {'name': '', 'service': 'S3', 'notes': ''}

ID: 4
Atributos: {'name': '', 'service': 'S3', 'notes': ''}

ID: 5
Atributos: {'name': '', 'service': 'WAF', 'notes': ''}

ID: 6
Atributos: {'name': 'Apache Metron', 'service': 'ThirdParty', 'notes': 'NAME: Apache Metron'}

ID: 7
Atributos: {'name': '', 'service': 'SNS', 'notes': ''}

ID: 8
Atributos: {'name': '', 'service': 'SQS', 'notes': ''}

ID: 9
Atributos: {'name': '', 'service': 'CloudTrail', 'notes': ''}


--- PRIMERAS 10 ARISTAS ---
('0', '6', {'flow_id': 0, 'notes': '', 'seq': '0', 'type': 'data', 'id': '0'})
('1', '4', {'flow_id': 1, 'notes': '', 'seq': '1', 'type': 'data', 'id': '0'})


In [10]:
from collections import Counter
import pandas as pd

service_counter = Counter()
third_party_names = Counter()
edge_type_counter = Counter()

for file_path in files:
    G = nx.read_graphml(file_path)

    for node_id, attrs in G.nodes(data=True):
        service = str(attrs.get("service", "")).strip()
        name = str(attrs.get("name", "")).strip()

        service_counter[service if service else "EMPTY_SERVICE"] += 1

        if service.lower() == "thirdparty":
            third_party_names[name if name else "UNNAMED_THIRDPARTY"] += 1

    for u, v, attrs in G.edges(data=True):
        edge_type = str(attrs.get("type", "")).strip()
        edge_type_counter[edge_type if edge_type else "EMPTY_TYPE"] += 1


print("Servicios más frecuentes:")
display(
    pd.DataFrame(
        service_counter.most_common(30),
        columns=["service", "count"]
    )
)

print("\nNombres encontrados en nodos ThirdParty:")
display(
    pd.DataFrame(
        third_party_names.most_common(30),
        columns=["third_party_name", "count"]
    )
)

print("\nTipos de aristas:")
display(
    pd.DataFrame(
        edge_type_counter.most_common(),
        columns=["edge_type", "count"]
    )
)

Servicios más frecuentes:


,service,count
0,Lambda,347
1,S3,335
2,ThirdParty,240
3,EC2,232
4,DynamoDB,129
5,ApiGateway,99
6,UserConsumerWeb,91
7,SQS,88
8,RDS,84
9,EKS,77



Nombres encontrados en nodos ThirdParty:


,third_party_name,count
0,Kafka,11
1,Slack,4
2,Salesforce,3
3,Elastisearch,3
4,Airflow,3
5,MySQL,2
6,SQL Server,2
7,Git,2
8,SAP,2
9,Dashboard,2



Tipos de aristas:


,edge_type,count
0,data,3609
1,meta,831


In [11]:
empty_services = []
empty_edge_types = []

for file_path in files:
    G = nx.read_graphml(file_path)
    graph_name = os.path.basename(file_path)

    for node_id, attrs in G.nodes(data=True):
        service = str(attrs.get("service", "")).strip()

        if not service:
            empty_services.append({
                "graph": graph_name,
                "node_id": node_id,
                "attributes": attrs
            })

    for source, target, attrs in G.edges(data=True):
        edge_type = str(attrs.get("type", "")).strip()

        if not edge_type:
            empty_edge_types.append({
                "graph": graph_name,
                "source": source,
                "target": target,
                "attributes": attrs
            })

print("Nodos sin service:", len(empty_services))
print("Aristas sin type:", len(empty_edge_types))

if empty_services:
    display(pd.DataFrame(empty_services).head(10))

if empty_edge_types:
    display(pd.DataFrame(empty_edge_types).head(10))

Nodos sin service: 0
Aristas sin type: 0


In [12]:
# Tomamos el primer grafo
file_path = files[0]
G = nx.read_graphml(file_path)

# Crear catálogos numéricos para este grafo de prueba
services = sorted({
    str(attrs["service"]).strip()
    for _, attrs in G.nodes(data=True)
})

edge_types = sorted({
    str(attrs["type"]).strip()
    for _, _, attrs in G.edges(data=True)
})

service_to_id = {
    service: i
    for i, service in enumerate(services)
}

edge_type_to_id = {
    edge_type: i
    for i, edge_type in enumerate(edge_types)
}

# Convertir los identificadores originales de los nodos
# a identificadores consecutivos: 0, 1, 2, ...
node_to_local_id = {
    node: i
    for i, node in enumerate(G.nodes())
}

print("CATÁLOGO DE SERVICIOS")
print(service_to_id)

print("\nCATÁLOGO DE TIPOS DE ARISTA")
print(edge_type_to_id)

print("\nFORMATO GSPAN")
print("t # 0")

for node, attrs in G.nodes(data=True):
    local_id = node_to_local_id[node]
    service = str(attrs["service"]).strip()
    service_id = service_to_id[service]

    print(f"v {local_id} {service_id}")

for source, target, attrs in G.edges(data=True):
    source_id = node_to_local_id[source]
    target_id = node_to_local_id[target]

    edge_type = str(attrs["type"]).strip()
    edge_type_id = edge_type_to_id[edge_type]

    print(f"e {source_id} {target_id} {edge_type_id}")

CATÁLOGO DE SERVICIOS
{'Athena': 0, 'CloudTrail': 1, 'EC2': 2, 'EMR': 3, 'GuardDuty': 4, 'OnPremDC': 5, 'S3': 6, 'SNS': 7, 'SQS': 8, 'ThirdParty': 9, 'UserCompanyAnalyst': 10, 'WAF': 11}

CATÁLOGO DE TIPOS DE ARISTA
{'data': 0, 'meta': 1}

FORMATO GSPAN
t # 0
v 0 5
v 1 3
v 2 0
v 3 6
v 4 6
v 5 11
v 6 9
v 7 7
v 8 8
v 9 1
v 10 4
v 11 2
v 12 10
e 0 6 0
e 1 4 0
e 2 6 0
e 3 6 0
e 4 2 0
e 5 11 1
e 6 1 0
e 6 11 0
e 7 6 0
e 8 6 0
e 9 6 0
e 10 6 0
e 11 6 1
e 11 5 0
e 11 12 0
e 12 11 1


In [13]:
all_services = set()
all_edge_types = set()

for file_path in files:
    G = nx.read_graphml(file_path)

    for _, attrs in G.nodes(data=True):
        service = str(attrs["service"]).strip()
        all_services.add(service)

    for _, _, attrs in G.edges(data=True):
        edge_type = str(attrs["type"]).strip()
        all_edge_types.add(edge_type)


# Catálogo único para todos los grafos
global_service_to_id = {
    service: index
    for index, service in enumerate(sorted(all_services))
}

global_edge_type_to_id = {
    edge_type: index
    for index, edge_type in enumerate(sorted(all_edge_types))
}


print("Cantidad de servicios diferentes:", len(global_service_to_id))
print("Catálogo global de aristas:", global_edge_type_to_id)

print("\nPrimeros 30 servicios del catálogo global:")

for service, service_id in list(global_service_to_id.items())[:30]:
    print(f"{service_id:3d} -> {service}")

Cantidad de servicios diferentes: 169
Catálogo global de aristas: {'data': 0, 'meta': 1}

Primeros 30 servicios del catálogo global:
  0 -> ACM
  1 -> ALB
  2 -> AMI
  3 -> AWSConfig
  4 -> AccessAnalyzer
  5 -> AlexaForBusiness
  6 -> AmazonML
  7 -> AmazonMQ
  8 -> Amplify
  9 -> ApiGateway
 10 -> AppDiscovery
 11 -> AppStream
 12 -> AppSync
 13 -> Athena
 14 -> Aurora
 15 -> AutoScaling
 16 -> Batch
 17 -> BeanStalk
 18 -> Chime
 19 -> CloudFormation
 20 -> CloudFront
 21 -> CloudHSM
 22 -> CloudTrail
 23 -> CloudWatch
 24 -> CodeBuild
 25 -> CodeCommit
 26 -> CodeDeploy
 27 -> CodePipeline
 28 -> Cognito
 29 -> Comprehend


In [14]:
import json

NODE_LABELS_FILE = "/content/Cloudscape/gspan_node_labels.json"
EDGE_LABELS_FILE = "/content/Cloudscape/gspan_edge_labels.json"

# Invertimos los diccionarios:
# ID -> nombre legible
global_id_to_service = {
    service_id: service
    for service, service_id in global_service_to_id.items()
}

global_id_to_edge_type = {
    edge_id: edge_type
    for edge_type, edge_id in global_edge_type_to_id.items()
}

with open(NODE_LABELS_FILE, "w", encoding="utf-8") as f:
    json.dump(
        global_id_to_service,
        f,
        ensure_ascii=False,
        indent=2
    )

with open(EDGE_LABELS_FILE, "w", encoding="utf-8") as f:
    json.dump(
        global_id_to_edge_type,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Catálogo de nodos guardado en:", NODE_LABELS_FILE)
print("Catálogo de aristas guardado en:", EDGE_LABELS_FILE)

Catálogo de nodos guardado en: /content/Cloudscape/gspan_node_labels.json
Catálogo de aristas guardado en: /content/Cloudscape/gspan_edge_labels.json


In [15]:
GSPAN_DATA_FILE = "/content/Cloudscape/cloudscape_gspan.data"
GRAPH_NAMES_FILE = "/content/Cloudscape/gspan_graph_names.json"

graph_id_to_name = {}

with open(GSPAN_DATA_FILE, "w", encoding="utf-8") as output:

    for graph_id, file_path in enumerate(files):

        G = nx.read_graphml(file_path)

        graph_name = os.path.basename(file_path)
        graph_id_to_name[graph_id] = graph_name

        # Inicio del grafo
        output.write(f"t # {graph_id}\n")

        # IDs locales consecutivos para los nodos
        node_to_local_id = {
            node: local_id
            for local_id, node in enumerate(G.nodes())
        }

        # Nodos
        for node, attrs in G.nodes(data=True):

            local_id = node_to_local_id[node]
            service = str(attrs["service"]).strip()
            service_id = global_service_to_id[service]

            output.write(
                f"v {local_id} {service_id}\n"
            )

        # Aristas dirigidas
        for source, target, attrs in G.edges(data=True):

            source_id = node_to_local_id[source]
            target_id = node_to_local_id[target]

            edge_type = str(attrs["type"]).strip()
            edge_type_id = global_edge_type_to_id[edge_type]

            output.write(
                f"e {source_id} {target_id} {edge_type_id}\n"
            )

    # Marca de fin de la colección
    output.write("t # -1\n")


with open(GRAPH_NAMES_FILE, "w", encoding="utf-8") as f:
    json.dump(
        graph_id_to_name,
        f,
        ensure_ascii=False,
        indent=2
    )


print("Archivo gSpan creado en:", GSPAN_DATA_FILE)
print("Cantidad de grafos escritos:", len(graph_id_to_name))
print("Mapa de nombres guardado en:", GRAPH_NAMES_FILE)

Archivo gSpan creado en: /content/Cloudscape/cloudscape_gspan.data
Cantidad de grafos escritos: 396
Mapa de nombres guardado en: /content/Cloudscape/gspan_graph_names.json


In [16]:
# Mostrar las primeras 35 líneas del archivo gSpan
with open(GSPAN_DATA_FILE, "r", encoding="utf-8") as f:
    primeras_lineas = [next(f).rstrip() for _ in range(35)]

print("\n".join(primeras_lineas))


# Verificar la última línea
with open(GSPAN_DATA_FILE, "r", encoding="utf-8") as f:
    ultima_linea = ""

    for linea in f:
        if linea.strip():
            ultima_linea = linea.strip()

print("\nÚltima línea:", ultima_linea)

t # 0
v 0 94
v 1 51
v 2 13
v 3 110
v 4 110
v 5 166
v 6 129
v 7 113
v 8 114
v 9 22
v 10 63
v 11 45
v 12 137
e 0 6 0
e 1 4 0
e 2 6 0
e 3 6 0
e 4 2 0
e 5 11 1
e 6 1 0
e 6 11 0
e 7 6 0
e 8 6 0
e 9 6 0
e 10 6 0
e 11 6 1
e 11 5 0
e 11 12 0
e 12 11 1
t # 1
v 0 49
v 1 129
v 2 129
v 3 42

Última línea: t # -1


In [17]:
!pip install gspan-mining

In [18]:
from pathlib import Path
import gspan_mining

gspan_file = (
    Path(gspan_mining.__file__).parent
    / "gspan.py"
)

codigo = gspan_file.read_text(encoding="utf-8")

# Reemplazar el método antiguo de pandas
codigo_corregido = codigo.replace(
    "self._report_df.append(",
    "self._report_df._append("
)

if codigo == codigo_corregido:
    print("No se encontró la línea que debía corregirse.")
else:
    gspan_file.write_text(
        codigo_corregido,
        encoding="utf-8"
    )
    print("Corrección aplicada en:")
    print(gspan_file)

Corrección aplicada en:
/usr/local/lib/python3.12/dist-packages/gspan_mining/gspan.py


##Flags

"-s" minimum number of nodes per graph (396)

"-l" minimum number of nodes per pattern

"-u" maximum number of nodes per pattern

"-d" directed graphs

"-w" show which graphs it appears in

In [19]:
import subprocess

command = [
    "python",
    "-m",
    "gspan_mining",
    "-s", "10",          # mínimo 40 de 396 grafos
    "-l", "2",           # mínimo 2 nodos por patrón
    "-u", "4",           # máximo 3 nodos por patrón
    "-d", "True",        # grafos dirigidos
    "-w", "True",        # mostrar en qué grafos aparece
    GSPAN_DATA_FILE
]

result = subprocess.run(
    command,
    capture_output=True,
    text=True
)

print("Código de salida:", result.returncode)
print("Cantidad de caracteres generados:", len(result.stdout))

if result.stderr:
    print("\nMensajes de error:")
    print(result.stderr[:3000])

print("\nPrimeras 80 líneas del resultado:")
print("\n".join(result.stdout.splitlines()[:80]))

Código de salida: 1
Cantidad de caracteres generados: 12466

Mensajes de error:


Primeras 80 líneas del resultado:
t # 0
v 0 51
v 1 110
e 0 1 0

Support: 14
where: [0, 225, 101, 102, 325, 171, 205, 113, 117, 343, 57, 187, 252, 125]

-----------------

t # 1
v 0 110
v 1 129
e 0 1 0

Support: 13
where: [0, 1, 256, 3, 313, 328, 271, 208, 243, 153, 187, 316, 255]

-----------------

t # 2
v 0 110
v 1 13
e 0 1 0

Support: 27
where: [0, 129, 3, 259, 389, 135, 11, 268, 272, 275, 277, 157, 163, 38, 303, 179, 61, 208, 212, 340, 342, 349, 225, 101, 113, 117, 381]

-----------------

t # 3
v 0 110
v 1 13
e 0 1 0
e 1 0 1

Support: 14
where: [129, 225, 3, 101, 38, 135, 389, 268, 303, 272, 113, 340, 277, 61]

-----------------

t # 4
v 0 129
v 1 45
e 0 1 0

Support: 17
where: [0, 64, 193, 226, 4, 197, 229, 294, 359, 329, 299, 147, 284, 119, 183, 156, 343]

-----------------

t # 5
v 0 49
v 1 129
e 0 1 0

Support: 12
where: [1, 226, 328, 331, 46, 277, 310, 183, 248, 57, 27, 349]

-----------------



In [20]:
import re
import pandas as pd

# ==========================================================
# AUTOMATICALLY TRANSLATE GSPAN OUTPUT
# ==========================================================

def obtener_catalogo(catalogo, clave):
    """
    Permite consultar catálogos cuyas claves sean enteros
    o cadenas, por ejemplo 76 o "76".
    """
    return catalogo.get(
        clave,
        catalogo.get(str(clave), f"UNKNOWN_{clave}")
    )


def parsear_salida_gspan(
    texto,
    catalogo_nodos,
    catalogo_aristas,
    catalogo_grafos=None,
    total_grafos=396
):
    """
    Convierte result.stdout en un DataFrame legible.
    """

    patrones = []
    patron_actual = None

    for linea_original in texto.splitlines():

        linea = linea_original.strip()

        if not linea:
            continue

        # Inicio de un patrón
        if linea.startswith("t #"):

            # Guardar el patrón anterior
            if patron_actual is not None:
                patrones.append(patron_actual)

            partes = linea.split()
            pattern_id = int(partes[-1])

            patron_actual = {
                "pattern_id": pattern_id,
                "nodes": {},
                "edges": [],
                "support": None,
                "graph_ids": []
            }

        # Nodo: v node_id label_id
        elif linea.startswith("v ") and patron_actual is not None:

            partes = linea.split()

            node_id = int(partes[1])
            label_id = int(partes[2])

            patron_actual["nodes"][node_id] = {
                "label_id": label_id,
                "label": obtener_catalogo(
                    catalogo_nodos,
                    label_id
                )
            }

        # Arista: e source target edge_label_id
        elif linea.startswith("e ") and patron_actual is not None:

            partes = linea.split()

            source = int(partes[1])
            target = int(partes[2])
            edge_label_id = int(partes[3])

            patron_actual["edges"].append({
                "source": source,
                "target": target,
                "label_id": edge_label_id,
                "label": obtener_catalogo(
                    catalogo_aristas,
                    edge_label_id
                )
            })

        # Soporte
        elif linea.startswith("Support:") and patron_actual is not None:

            patron_actual["support"] = int(
                linea.split(":", 1)[1].strip()
            )

        # IDs de los grafos donde aparece
        elif linea.startswith("where:") and patron_actual is not None:

            patron_actual["graph_ids"] = [
                int(valor)
                for valor in re.findall(r"\d+", linea)
            ]

    # Guardar el último patrón
    if patron_actual is not None:
        patrones.append(patron_actual)

    filas = []

    for patron in patrones:

        conexiones = []

        for edge in patron["edges"]:

            source_label = patron["nodes"][
                edge["source"]
            ]["label"]

            target_label = patron["nodes"][
                edge["target"]
            ]["label"]

            conexiones.append(
                f"{source_label} "
                f"-[{edge['label']}]-> "
                f"{target_label}"
            )

        graph_ids = patron["graph_ids"]

        # Traducir IDs de grafos a nombres de archivos
        if catalogo_grafos is not None:
            arquitecturas = [
                obtener_catalogo(
                    catalogo_grafos,
                    graph_id
                )
                for graph_id in graph_ids
            ]
        else:
            arquitecturas = graph_ids

        support = patron["support"]

        filas.append({
            "pattern_id": patron["pattern_id"],
            "pattern": " | ".join(conexiones),
            "num_nodes": len(patron["nodes"]),
            "num_edges": len(patron["edges"]),
            "support": support,
            "percentage_graphs": (
                100 * support / total_grafos
                if support is not None
                else None
            ),
            "graph_ids": graph_ids,
            "architectures": arquitecturas
        })

    df = pd.DataFrame(filas)

    if not df.empty:
        df = df.sort_values(
            by=["num_edges", "support", "pattern"],
            ascending=[True, False, True]
        ).reset_index(drop=True)

    return df


# Procesar toda la salida almacenada en result.stdout
df_gspan_patterns = parsear_salida_gspan(
    texto=result.stdout,
    catalogo_nodos=global_id_to_service,
    catalogo_aristas=global_id_to_edge_type,
    catalogo_grafos=graph_id_to_name,
    total_grafos=len(graph_id_to_name)
)

print("Patrones encontrados:", len(df_gspan_patterns))

display(
    df_gspan_patterns[
        [
            "pattern_id",
            "pattern",
            "num_nodes",
            "num_edges",
            "support",
            "percentage_graphs"
        ]
    ]
)

Patrones encontrados: 84


,pattern_id,pattern,num_nodes,num_edges,support,percentage_graphs
0,14,Lambda -[data]-> DynamoDB,2,1,56,14.141414
1,27,ApiGateway -[data]-> Lambda,2,1,45,11.363636
2,43,Lambda -[data]-> S3,2,1,43,10.858586
3,68,S3 -[data]-> Lambda,2,1,40,10.101010
4,28,DynamoDB -[data]-> Lambda,2,1,37,9.343434
...,...,...,...,...,...,...
79,36,Lambda -[data]-> ApiGateway | ApiGateway -[dat...,2,2,10,2.525253
80,58,S3 -[data]-> CloudFront | CloudFront -[meta]-> S3,2,2,10,2.525253
81,18,S3 -[data]-> EMR | EMR -[meta]-> S3,2,2,10,2.525253
82,79,S3 -[meta]-> Lambda | Lambda -[data]-> S3,2,2,10,2.525253


In [21]:
fila = df_gspan_patterns[
    df_gspan_patterns["pattern"] == "Lambda -[data]-> DynamoDB"
].iloc[0]

print("IDs de arquitecturas:")
print(fila["graph_ids"])

print("\nNombres de archivos:")
print(fila["architectures"])

IDs de arquitecturas:
[129, 255, 261, 7, 136, 391, 10, 138, 266, 144, 272, 273, 275, 150, 279, 281, 26, 287, 35, 292, 40, 44, 304, 177, 180, 184, 186, 61, 62, 194, 195, 201, 76, 335, 210, 84, 216, 93, 222, 350, 351, 98, 377, 369, 242, 243, 371, 372, 373, 249, 378, 251, 124, 253, 382, 127]

Nombres de archivos:
['KM5ONS2fnG0.graphml', 'co4_t2gN1T8.graphml', 'eEfWd4EgH_s.graphml', '0JxJpNjI9Y0.graphml', 'KywvGM6HVXI.graphml', 'yXpd8gPfows.graphml', '1SwHH7qQ6Pc.graphml', 'LD5ksgnu8r8.graphml', 'fRv8sOUyhZw.graphml', 'MbkLJ62jtMc.graphml', 'gcgtLDB0cKA.graphml', 'gpWR5JBC64A.graphml', 'h0HE3bOEiMk.graphml', 'OQKOHNtyz3E.graphml', 'hbz63Ve-eIY.graphml', 'hlVnmCfydIs.graphml', '4zVB5RbSTCo.graphml', 'iSkWd31X7zo.graphml', '66h7DrOEF5k.graphml', 'iwDNnyiD26M.graphml', '6iK4WNj6QqI.graphml', '7V8wTCkjOqo.graphml', 'kszaIJEnKPk.graphml', 'SpIlpGxuwFM.graphml', 'T048vs9p1h4.graphml', 'U8xlmurP-6o.graphml', 'UsngU-HjH_Q.graphml', 'AzM_d7ZvzUE.graphml', 'BPvr0qWpJlA.graphml', 'WtCfHP6rUAY.graphml

In [22]:
df_arquitecturas_patron = pd.DataFrame({
    "graph_id": fila["graph_ids"],
    "graph_name": fila["architectures"]
})

display(df_arquitecturas_patron)

,graph_id,graph_name
0,129,KM5ONS2fnG0.graphml
1,255,co4_t2gN1T8.graphml
2,261,eEfWd4EgH_s.graphml
3,7,0JxJpNjI9Y0.graphml
4,136,KywvGM6HVXI.graphml
5,391,yXpd8gPfows.graphml
6,10,1SwHH7qQ6Pc.graphml
7,138,LD5ksgnu8r8.graphml
8,266,fRv8sOUyhZw.graphml
9,144,MbkLJ62jtMc.graphml


In [23]:
import os
import pandas as pd

# ==========================================================
# DIFERENCIAR LOS PATRONES GSPAN POR CATEGORÍA
# ==========================================================

# ----------------------------------------------------------
# 1. Normalizar nombres de arquitectura
# ----------------------------------------------------------
def normalize_architecture_name(value):
    """
    Convierte:
        abc.graphml -> abc
        /ruta/abc.graphml -> abc

    Esto permite relacionar los nombres producidos por gSpan
    con la columna 'architecture' del CSV.
    """
    if pd.isna(value):
        return ""

    value = os.path.basename(str(value).strip())
    value = os.path.splitext(value)[0]

    return value


# Nombre normalizado en el CSV de categorías
df["architecture_normalized"] = (
    df["architecture"]
    .apply(normalize_architecture_name)
)

# ----------------------------------------------------------
# 2. Crear conjuntos de arquitecturas por categoría
# ----------------------------------------------------------

category_architectures = {
    "Edge": set(
        df.loc[
            df["tipo_arquitectura"].isin(["Edge", "Edge+HPC"]),
            "architecture_normalized"
        ]
    ),

    "HPC": set(
        df.loc[
            df["tipo_arquitectura"].isin(["HPC", "Edge+HPC"]),
            "architecture_normalized"
        ]
    ),

    #"Mixto": set(
    #    df.loc[
    #        df["tipo_arquitectura"] == "Edge+HPC",
    #        "architecture_normalized"
    #    ]
    #),

    "Serverless": set(
        df.loc[
            df["belongs_serverless"],
            "architecture_normalized"
        ]
    ),

    "None": set(
        df.loc[
            df["belongs_none_analytical"],
            "architecture_normalized"
        ]
    )
}

print("Arquitecturas por categoría:")

for category, architectures_set in category_architectures.items():
    print(f"{category}: {len(architectures_set)}")


# ----------------------------------------------------------
# 3. Calcular presencia de cada patrón en cada categoría
# ----------------------------------------------------------

rows_by_category = []

for _, pattern_row in df_gspan_patterns.iterrows():

    # Arquitecturas donde gSpan encontró el patrón
    pattern_architectures = {
        normalize_architecture_name(name)
        for name in pattern_row["architectures"]
    }

    for category, category_set in category_architectures.items():

        # Intersección:
        # arquitecturas de la categoría que contienen el patrón
        matching_architectures = (
            pattern_architectures
            & category_set
        )

        category_total = len(category_set)
        category_support = len(matching_architectures)

        category_percentage = (
            100 * category_support / category_total
            if category_total > 0
            else 0
        )

        rows_by_category.append({
            "pattern_id": pattern_row["pattern_id"],
            "pattern": pattern_row["pattern"],
            "category": category,

            # Soporte global original
            "global_support": pattern_row["support"],
            "global_percentage": pattern_row["percentage_graphs"],

            # Soporte dentro de la categoría
            "category_support": category_support,
            "category_total": category_total,
            "category_percentage": category_percentage,

            # Arquitecturas concretas de esa categoría
            "architectures": sorted(matching_architectures)
        })


df_patterns_by_category = pd.DataFrame(rows_by_category)

df_patterns_by_category = (
    df_patterns_by_category
    .sort_values(
        by=[
            "category",
            "category_percentage",
            "category_support"
        ],
        ascending=[True, False, False]
    )
    .reset_index(drop=True)
)

display(
    df_patterns_by_category[
        [
            "pattern",
            "category",
            "category_support",
            "category_total",
            "category_percentage",
            "global_support"
        ]
    ]
)

Arquitecturas por categoría:
Edge: 106
HPC: 15
Serverless: 352
None: 40


,pattern,category,category_support,category_total,category_percentage,global_support
0,Lambda -[data]-> DynamoDB,Edge,15,106,14.150943,56
1,S3 -[data]-> CloudFront,Edge,15,106,14.150943,15
2,Lambda -[data]-> S3,Edge,13,106,12.264151,43
3,Firehose -[data]-> S3,Edge,13,106,12.264151,24
4,CloudFront -[data]-> S3,Edge,13,106,12.264151,13
...,...,...,...,...,...,...
331,EKS -[data]-> EC2,Serverless,9,352,2.556818,10
332,ThirdParty -[data]-> EKS,Serverless,9,352,2.556818,10
333,ALB -[data]-> EC2,Serverless,8,352,2.272727,10
334,EC2 -[data]-> ElastiCache,Serverless,8,352,2.272727,10


#Patterns By Category using gSpan

In [30]:
df_final = pd.read_csv(
    "/content/classified_architectures_with_metadata.csv"
)

print("Filas:", len(df_final))
print("Arquitecturas únicas:", df_final["architecture"].nunique())
print(df_final.columns.tolist())

display(df_final.head())

Filas: 396
Arquitecturas únicas: 396
['architecture', 'name', 'description', 'category', 'link', 'tipo_arquitectura', 'tipo_arquitectura_no_cloudfront', 'services', 'serverless_found', 'num_serverless', 'has_serverless', 'serverless_faas_found', 'num_serverless_faas', 'has_serverless_faas', 'serverless_compute_found', 'num_serverless_compute', 'has_serverless_compute', 'serverless_event_driven_found', 'num_serverless_event_driven', 'has_serverless_event_driven', 'belongs_edge', 'belongs_hpc', 'belongs_serverless', 'belongs_serverless_faas', 'belongs_serverless_compute', 'belongs_serverless_event_driven', 'belongs_none_analytical', 'belongs_edge_no_cloudfront']


,architecture,name,description,category,link,tipo_arquitectura,tipo_arquitectura_no_cloudfront,services,serverless_found,num_serverless,...,num_serverless_event_driven,has_serverless_event_driven,belongs_edge,belongs_hpc,belongs_serverless,belongs_serverless_faas,belongs_serverless_compute,belongs_serverless_event_driven,belongs_none_analytical,belongs_edge_no_cloudfront
0,j-lPgPGBTwQ,Halter:Building the Future of Farming with IoT...,This product has two flows (mainly). The first...,data_ingestion,https://www.youtube.com/watch?v=j-lPgPGBTwQ,Edge,Edge,"['Lambda', 'S3', 'ThirdParty', 'IoTCore', 'Use...","['Fargate', 'Lambda', 'S3']",3,...,0,False,True,False,True,True,True,False,False,True
1,Gds8hl8dKuo,Fugue: Staying in Compliance by Automating Dri...,Infra provisioning to help businesses build in...,control,https://www.youtube.com/watch?v=Gds8hl8dKuo,NaN,NaN,"['SQS', 'KMS', 'UserCompanyAgent', 'SNS', 'Dyn...","['DynamoDB', 'S3', 'SNS', 'SQS']",4,...,1,True,False,False,True,False,False,True,False,False
2,j9OZ-7aCAyA,Homesite: Event-Driven Data Analytics Platform...,Step functions - some actions would exceed Lam...,data_ingestion,https://www.youtube.com/watch?v=j9OZ-7aCAyA,NaN,NaN,"['SQS', 'Glue', 'StepFunctions', 'Lambda', 'Us...","['DynamoDB', 'Lambda', 'S3', 'SQS', 'StepFunct...",5,...,0,False,False,False,True,True,False,False,False,False
3,Eoq7E6jMtBs,Orangetheory Fitness: Using Chime SDK to Power...,fitness studio providing coach led classes. OT...,interactive,https://www.youtube.com/watch?v=Eoq7E6jMtBs,NaN,NaN,"['UserConsumerMobile', 'Lambda', 'Cognito', 'A...","['APIGateway', 'DynamoDB', 'Lambda', 'S3']",4,...,0,False,False,False,True,True,False,False,False,False
4,1kWxymroGeE,OutSystems: Decomposing a Data Monolith for Sc...,Data is key to monolith decomposition. This vi...,control,https://www.youtube.com/watch?v=1kWxymroGeE,NaN,NaN,"['KMS', 'S3', 'DynamoDB', 'Aurora', 'SecretsMa...","['DynamoDB', 'S3']",2,...,0,False,False,False,True,False,False,False,False,False


In [31]:
import ast
import re
import pandas as pd

# ==========================================================
# CREAR FaaS, SERVERLESS COMPUTE Y EVENT-DRIVEN EN df_final
# ==========================================================

def parse_services_local(x):
    if isinstance(x, list):
        return [str(v).strip() for v in x]

    if pd.isna(x) or str(x).strip() == "":
        return []

    s = str(x).strip()

    try:
        value = ast.literal_eval(s)
        if isinstance(value, list):
            return [str(v).strip() for v in value]
    except Exception:
        pass

    return [
        item.strip()
        for item in s.split(",")
        if item.strip()
    ]


def canonical_service_name_local(s):
    s = str(s).strip()
    s = re.sub(r"\s+", " ", s)

    compact = re.sub(
        r"[^A-Za-z0-9]+",
        "",
        s
    ).lower()

    alias_map = {
        "lambda": "Lambda",
        "lambdaatedge": "LambdaAtEdge",
        "fargate": "Fargate",
        "eventbridge": "EventBridge",
        "sns": "SNS"
    }

    return alias_map.get(compact, s)


serverless_faas = [
    "Lambda",
    "LambdaAtEdge"
]

serverless_compute = [
    "Fargate"
]

serverless_event_driven = [
    "EventBridge",
    "SNS"
]


serverless_faas_canonical = {
    canonical_service_name_local(s)
    for s in serverless_faas
}

serverless_compute_canonical = {
    canonical_service_name_local(s)
    for s in serverless_compute
}

serverless_event_driven_canonical = {
    canonical_service_name_local(s)
    for s in serverless_event_driven
}


df_final = df_final.copy()

df_final["services_list_tmp"] = (
    df_final["services"]
    .apply(parse_services_local)
)

df_final["services_canonical_tmp"] = (
    df_final["services_list_tmp"]
    .apply(
        lambda lst: sorted(
            set(
                canonical_service_name_local(x)
                for x in lst
            )
        )
    )
)


df_final["faas_found"] = (
    df_final["services_canonical_tmp"]
    .apply(
        lambda lst: sorted(
            set(
                x for x in lst
                if x in serverless_faas_canonical
            )
        )
    )
)

df_final["num_faas"] = (
    df_final["faas_found"]
    .apply(len)
)

df_final["has_faas"] = (
    df_final["num_faas"] > 0
)


df_final["serverless_compute_found"] = (
    df_final["services_canonical_tmp"]
    .apply(
        lambda lst: sorted(
            set(
                x for x in lst
                if x in serverless_compute_canonical
            )
        )
    )
)

df_final["num_serverless_compute"] = (
    df_final["serverless_compute_found"]
    .apply(len)
)

df_final["has_serverless_compute"] = (
    df_final["num_serverless_compute"] > 0
)


df_final["event_driven_found"] = (
    df_final["services_canonical_tmp"]
    .apply(
        lambda lst: sorted(
            set(
                x for x in lst
                if x in serverless_event_driven_canonical
            )
        )
    )
)

df_final["num_event_driven"] = (
    df_final["event_driven_found"]
    .apply(len)
)

df_final["has_event_driven"] = (
    df_final["num_event_driven"] > 0
)


print("Columnas creadas correctamente:")
print(
    df_final[
        [
            "has_faas",
            "has_serverless_compute",
            "has_event_driven"
        ]
    ].sum()
)

Columnas creadas correctamente:
has_faas                  224
has_serverless_compute     29
has_event_driven           64
dtype: int64


In [32]:
#subset_faas
#subset_serverless_compute
#subset_event_driven

In [33]:
category_architectures = {
    "Edge": set(
        df_final.loc[
            df_final["belongs_edge"],
            "architecture"
        ].astype(str)
    ),

    "Edge no CloudFront": set(
        df_final.loc[
            df_final["belongs_edge_no_cloudfront"],
            "architecture"
        ].astype(str)
    ),

    "HPC": set(
        df_final.loc[
            df_final["belongs_hpc"],
            "architecture"
        ].astype(str)
    ),

    "Serverless": set(
        df_final.loc[
            df_final["belongs_serverless"],
            "architecture"
        ].astype(str)
    ),

    "FaaS": set(
        df_final.loc[
            df_final["has_faas"],
            "architecture"
        ].astype(str)
    ),

    "Serverless Compute": set(
        df_final.loc[
            df_final["has_serverless_compute"],
            "architecture"
        ].astype(str)
    ),

    "Event-Driven": set(
        df_final.loc[
            df_final["has_event_driven"],
            "architecture"
        ].astype(str)
    ),

    "None": set(
        df_final.loc[
            df_final["belongs_none_analytical"],
            "architecture"
        ].astype(str)
    )
}

print("Arquitecturas por categoría:\n")

for category, architectures in category_architectures.items():
    print(f"{category}: {len(architectures)}")

Arquitecturas por categoría:

Edge: 106
Edge no CloudFront: 77
HPC: 15
Serverless: 352
FaaS: 224
Serverless Compute: 29
Event-Driven: 64
None: 40


In [34]:
import os
import json
import networkx as nx

CATEGORY_GSPAN_DIR = "/content/Cloudscape/gspan_categories"
os.makedirs(CATEGORY_GSPAN_DIR, exist_ok=True)


def normalize_architecture_name(value):
    value = os.path.basename(str(value).strip())
    return os.path.splitext(value)[0]


def create_category_gspan_file(
    category_name,
    architecture_names,
    graph_files,
    service_catalog,
    edge_catalog
):
    """
    Crea un archivo gSpan únicamente con las arquitecturas
    pertenecientes a una categoría analítica.
    """

    architecture_names = {
        normalize_architecture_name(name)
        for name in architecture_names
    }

    safe_category_name = (
        category_name.lower()
        .replace(" ", "_")
        .replace("-", "_")
    )

    output_path = os.path.join(
        CATEGORY_GSPAN_DIR,
        f"{safe_category_name}_gspan.data"
    )

    mapping_path = os.path.join(
        CATEGORY_GSPAN_DIR,
        f"{safe_category_name}_graph_names.json"
    )

    category_graph_id_to_name = {}
    category_graph_id = 0

    with open(output_path, "w", encoding="utf-8") as output:

        for file_path in graph_files:

            graph_name = os.path.basename(file_path)
            architecture_id = normalize_architecture_name(graph_name)

            # Saltar grafos que no pertenecen a esta categoría
            if architecture_id not in architecture_names:
                continue

            G = nx.read_graphml(file_path)

            category_graph_id_to_name[category_graph_id] = graph_name

            output.write(f"t # {category_graph_id}\n")

            node_to_local_id = {
                node: local_id
                for local_id, node in enumerate(G.nodes())
            }

            # Escribir nodos
            for node, attrs in G.nodes(data=True):

                local_id = node_to_local_id[node]
                service = str(attrs["service"]).strip()
                service_id = service_catalog[service]

                output.write(
                    f"v {local_id} {service_id}\n"
                )

            # Escribir aristas
            for source, target, attrs in G.edges(data=True):

                source_id = node_to_local_id[source]
                target_id = node_to_local_id[target]

                edge_type = str(attrs["type"]).strip()
                edge_type_id = edge_catalog[edge_type]

                output.write(
                    f"e {source_id} {target_id} {edge_type_id}\n"
                )

            category_graph_id += 1

        output.write("t # -1\n")

    with open(mapping_path, "w", encoding="utf-8") as file:
        json.dump(
            category_graph_id_to_name,
            file,
            ensure_ascii=False,
            indent=2
        )

    return {
        "category": category_name,
        "gspan_file": output_path,
        "mapping_file": mapping_path,
        "num_graphs": category_graph_id,
        "graph_id_to_name": category_graph_id_to_name
    }

In [35]:
category_gspan_files = {}

for category_name, architecture_set in category_architectures.items():

    info = create_category_gspan_file(
        category_name=category_name,
        architecture_names=architecture_set,
        graph_files=files,
        service_catalog=global_service_to_id,
        edge_catalog=global_edge_type_to_id
    )

    category_gspan_files[category_name] = info

    print(
        f"{category_name}: "
        f"{info['num_graphs']} grafos -> "
        f"{info['gspan_file']}"
    )

Edge: 106 grafos -> /content/Cloudscape/gspan_categories/edge_gspan.data
Edge no CloudFront: 77 grafos -> /content/Cloudscape/gspan_categories/edge_no_cloudfront_gspan.data
HPC: 15 grafos -> /content/Cloudscape/gspan_categories/hpc_gspan.data
Serverless: 352 grafos -> /content/Cloudscape/gspan_categories/serverless_gspan.data
FaaS: 224 grafos -> /content/Cloudscape/gspan_categories/faas_gspan.data
Serverless Compute: 29 grafos -> /content/Cloudscape/gspan_categories/serverless_compute_gspan.data
Event-Driven: 64 grafos -> /content/Cloudscape/gspan_categories/event_driven_gspan.data
None: 40 grafos -> /content/Cloudscape/gspan_categories/none_gspan.data


"-s" minimum number of nodes per graph (396)

"-l" minimum number of nodes per pattern

"-u" maximum number of nodes per pattern

"-d" directed graphs

"-w" show which graphs it appears in

##Global catalog


---



##USERCONSUMER-USERCOMPANY Standardized


In [36]:
import os
import re
import json
import math
import glob
import subprocess

from pathlib import Path
from collections import Counter

import pandas as pd
import networkx as nx


# ==========================================================
# RUTAS PARA EL ANÁLISIS NORMALIZADO
# ==========================================================

GRAPH_DIR_NORM = "/content/Cloudscape/data/graphs"

METADATA_CSV_NORM = (
    "/content/classified_architectures_with_metadata.csv"
)

OUTPUT_DIR_NORM = (
    "/content/Cloudscape/gspan_normalized_by_category"
)

os.makedirs(OUTPUT_DIR_NORM, exist_ok=True)


# Lista independiente de GraphML.
files_norm = sorted(
    glob.glob(
        os.path.join(GRAPH_DIR_NORM, "*.graphml")
    )
)

if not files_norm:
    raise FileNotFoundError(
        f"No se encontraron archivos GraphML en {GRAPH_DIR_NORM}"
    )

print("GraphML encontrados:", len(files_norm))
print("Directorio de resultados:", OUTPUT_DIR_NORM)

GraphML encontrados: 396
Directorio de resultados: /content/Cloudscape/gspan_normalized_by_category


In [37]:
# ==========================================================
# NORMALIZACIÓN DE ETIQUETAS DE SERVICIO PARA GSPAN
# ==========================================================

def normalizar_servicio_gspan(valor):
    """
    Convierte variantes de usuarios/actores en etiquetas generales.

    Ejemplos:
        UserCompany123          -> UserCompany
        UserCompany_ABC         -> UserCompany
        User-Company-01         -> UserCompany
        UsserCompany123         -> UserCompany
        UserConsumer555         -> UserConsumer
        User Consumer XYZ       -> UserConsumer

    Los demás servicios permanecen sin cambios.
    """

    if valor is None:
        return "EMPTY_SERVICE"

    servicio = str(valor).strip()

    if not servicio or servicio.casefold() == "nan":
        return "EMPTY_SERVICE"

    # Elimina separadores como _, -, espacios, etc.,
    # para detectar variantes del mismo prefijo.
    servicio_compacto = re.sub(
        r"[^a-z0-9]+",
        "",
        servicio.casefold()
    )

    # Incluye "ussercompany" por si existen registros con ese typo.
    if servicio_compacto.startswith(
        ("usercompany", "ussercompany")
    ):
        return "UserCompany"

    if servicio_compacto.startswith("userconsumer"):
        return "UserConsumer"

    return servicio


def normalizar_tipo_arista_gspan(valor):
    """
    Evita problemas si alguna arista llegara a no tener tipo.
    """

    if valor is None:
        return "EMPTY_TYPE"

    edge_type = str(valor).strip()

    if not edge_type or edge_type.casefold() == "nan":
        return "EMPTY_TYPE"

    return edge_type

In [38]:
# ==========================================================
# AUDITORÍA DE LA NORMALIZACIÓN
# ==========================================================

raw_service_counter_norm = Counter()
normalized_service_counter_norm = Counter()
normalization_changes_norm = Counter()

for file_path_norm in files_norm:

    G_norm = nx.read_graphml(file_path_norm)

    for _, attrs_norm in G_norm.nodes(data=True):

        service_raw_norm = str(
            attrs_norm.get("service", "")
        ).strip()

        service_normalized_norm = (
            normalizar_servicio_gspan(
                service_raw_norm
            )
        )

        raw_service_counter_norm[
            service_raw_norm
            if service_raw_norm
            else "EMPTY_SERVICE"
        ] += 1

        normalized_service_counter_norm[
            service_normalized_norm
        ] += 1

        if service_raw_norm != service_normalized_norm:
            normalization_changes_norm[
                (
                    service_raw_norm,
                    service_normalized_norm
                )
            ] += 1


df_normalization_audit_norm = pd.DataFrame(
    [
        {
            "original_service": original_service,
            "normalized_service": normalized_service,
            "affected_nodes": count
        }
        for (
            original_service,
            normalized_service
        ), count in normalization_changes_norm.items()
    ]
)

if not df_normalization_audit_norm.empty:

    df_normalization_audit_norm = (
        df_normalization_audit_norm
        .sort_values(
            by=[
                "normalized_service",
                "affected_nodes",
                "original_service"
            ],
            ascending=[True, False, True]
        )
        .reset_index(drop=True)
    )

print(
    "Cantidad de servicios distintos originales:",
    len(raw_service_counter_norm)
)

print(
    "Cantidad de servicios distintos normalizados:",
    len(normalized_service_counter_norm)
)

print("\nVariantes agrupadas:")

display(df_normalization_audit_norm.head(100))

print("\nServicios normalizados más frecuentes:")

display(
    pd.DataFrame(
        normalized_service_counter_norm.most_common(30),
        columns=["service", "node_count"]
    )
)

Cantidad de servicios distintos originales: 169
Cantidad de servicios distintos normalizados: 143

Variantes agrupadas:


,original_service,normalized_service,affected_nodes
0,UserCompanyDataStream,UserCompany,44
1,UserCompanyDeveloper,UserCompany,39
2,UserCompanyAnalyst,UserCompany,30
3,UserCompanyAgent,UserCompany,21
4,UserCompanyAPI,UserCompany,14
5,UserCompanyEdge,UserCompany,5
6,UserCompanyInternalPlatform,UserCompany,5
7,UserCompanyCRM,UserCompany,4
8,UserCompanyDrone,UserCompany,2
9,UserCompanyWebsite,UserCompany,2



Servicios normalizados más frecuentes:


,service,node_count
0,Lambda,347
1,S3,335
2,UserConsumer,263
3,ThirdParty,240
4,EC2,232
5,UserCompany,169
6,DynamoDB,129
7,ApiGateway,99
8,SQS,88
9,RDS,84


In [39]:
# ==========================================================
# CATÁLOGOS GLOBALES NORMALIZADOS
# ==========================================================

all_services_norm = set()
all_edge_types_norm = set()

for file_path_norm in files_norm:

    G_norm = nx.read_graphml(file_path_norm)

    for _, attrs_norm in G_norm.nodes(data=True):

        service_norm = normalizar_servicio_gspan(
            attrs_norm.get("service", "")
        )

        all_services_norm.add(service_norm)

    for _, _, attrs_norm in G_norm.edges(data=True):

        edge_type_norm = normalizar_tipo_arista_gspan(
            attrs_norm.get("type", "")
        )

        all_edge_types_norm.add(edge_type_norm)


global_service_to_id_norm = {
    service: index
    for index, service in enumerate(
        sorted(all_services_norm)
    )
}

global_edge_type_to_id_norm = {
    edge_type: index
    for index, edge_type in enumerate(
        sorted(all_edge_types_norm)
    )
}

global_id_to_service_norm = {
    service_id: service
    for service, service_id
    in global_service_to_id_norm.items()
}

global_id_to_edge_type_norm = {
    edge_type_id: edge_type
    for edge_type, edge_type_id
    in global_edge_type_to_id_norm.items()
}


print(
    "Servicios distintos después de normalizar:",
    len(global_service_to_id_norm)
)

print(
    "Tipos de arista distintos:",
    len(global_edge_type_to_id_norm)
)

print("\nEtiquetas relevantes:")

for service_check_norm in [
    "UserCompany",
    "UserConsumer"
]:
    print(
        f"{service_check_norm}:",
        global_service_to_id_norm.get(
            service_check_norm,
            "NO ENCONTRADO"
        )
    )

print("\nPrimeros 40 servicios del catálogo:")

for service_norm, service_id_norm in list(
    global_service_to_id_norm.items()
)[:40]:
    print(f"{service_id_norm:3d} -> {service_norm}")

Servicios distintos después de normalizar: 143
Tipos de arista distintos: 2

Etiquetas relevantes:
UserCompany: 135
UserConsumer: 136

Primeros 40 servicios del catálogo:
  0 -> ACM
  1 -> ALB
  2 -> AMI
  3 -> AWSConfig
  4 -> AccessAnalyzer
  5 -> AlexaForBusiness
  6 -> AmazonML
  7 -> AmazonMQ
  8 -> Amplify
  9 -> ApiGateway
 10 -> AppDiscovery
 11 -> AppStream
 12 -> AppSync
 13 -> Athena
 14 -> Aurora
 15 -> AutoScaling
 16 -> Batch
 17 -> BeanStalk
 18 -> Chime
 19 -> CloudFormation
 20 -> CloudFront
 21 -> CloudHSM
 22 -> CloudTrail
 23 -> CloudWatch
 24 -> CodeBuild
 25 -> CodeCommit
 26 -> CodeDeploy
 27 -> CodePipeline
 28 -> Cognito
 29 -> Comprehend
 30 -> Connect
 31 -> ControlTower
 32 -> CouchBase
 33 -> DMS
 34 -> DataExchange
 35 -> DataPipeline
 36 -> DeepLens
 37 -> Detective
 38 -> DevTools
 39 -> DirectConnect


In [40]:
# ==========================================================
# GUARDAR CATÁLOGOS NORMALIZADOS
# ==========================================================

NODE_LABELS_FILE_NORM = os.path.join(
    OUTPUT_DIR_NORM,
    "gspan_node_labels_normalized.json"
)

EDGE_LABELS_FILE_NORM = os.path.join(
    OUTPUT_DIR_NORM,
    "gspan_edge_labels_normalized.json"
)

with open(
    NODE_LABELS_FILE_NORM,
    "w",
    encoding="utf-8"
) as file_norm:

    json.dump(
        global_id_to_service_norm,
        file_norm,
        ensure_ascii=False,
        indent=2
    )

with open(
    EDGE_LABELS_FILE_NORM,
    "w",
    encoding="utf-8"
) as file_norm:

    json.dump(
        global_id_to_edge_type_norm,
        file_norm,
        ensure_ascii=False,
        indent=2
    )

print("Catálogo normalizado de nodos:")
print(NODE_LABELS_FILE_NORM)

print("\nCatálogo normalizado de aristas:")
print(EDGE_LABELS_FILE_NORM)

Catálogo normalizado de nodos:
/content/Cloudscape/gspan_normalized_by_category/gspan_node_labels_normalized.json

Catálogo normalizado de aristas:
/content/Cloudscape/gspan_normalized_by_category/gspan_edge_labels_normalized.json


In [41]:
# ==========================================================
# CARGAR METADATOS Y DEFINIR CATEGORÍAS
# ==========================================================

df_metadata_norm = pd.read_csv(
    METADATA_CSV_NORM
).copy()


def normalizar_nombre_arquitectura_gspan(valor):
    """
    Convierte:
        /ruta/arquitectura.graphml -> arquitectura
        arquitectura.graphml       -> arquitectura
        arquitectura               -> arquitectura
    """

    if pd.isna(valor):
        return ""

    nombre = os.path.basename(
        str(valor).strip()
    )

    nombre = os.path.splitext(nombre)[0]

    return nombre


def convertir_a_booleano_gspan(serie):
    """
    Convierte de manera robusta una columna a booleanos.

    Funciona tanto si el CSV contiene:
        True / False
        "True" / "False"
        1 / 0
        "1" / "0"
    """

    if pd.api.types.is_bool_dtype(serie):
        return serie.fillna(False)

    valores_true = {
        "true",
        "1",
        "yes",
        "y",
        "si",
        "sí",
        "t"
    }

    return (
        serie
        .fillna(False)
        .astype(str)
        .str.strip()
        .str.casefold()
        .isin(valores_true)
    )


required_columns_norm = [
    "architecture",
    "belongs_hpc",
    "belongs_edge",
    "belongs_serverless",
    "belongs_none_analytical",
    "belongs_edge_no_cloudfront",
    "belongs_serverless_faas"
]

missing_columns_norm = [
    column_norm
    for column_norm in required_columns_norm
    if column_norm not in df_metadata_norm.columns
]

if missing_columns_norm:
    raise ValueError(
        "Faltan columnas necesarias en el CSV:\n"
        f"{missing_columns_norm}\n\n"
        "Columnas disponibles:\n"
        f"{df_metadata_norm.columns.tolist()}"
    )


df_metadata_norm["architecture_normalized_gspan"] = (
    df_metadata_norm["architecture"]
    .apply(normalizar_nombre_arquitectura_gspan)
)


category_architectures_norm = {
    "HPC": set(
        df_metadata_norm.loc[
            convertir_a_booleano_gspan(
                df_metadata_norm["belongs_hpc"]
            ),
            "architecture_normalized_gspan"
        ]
    ),

    "Edge": set(
        df_metadata_norm.loc[
            convertir_a_booleano_gspan(
                df_metadata_norm["belongs_edge"]
            ),
            "architecture_normalized_gspan"
        ]
    ),

    "Serverless": set(
        df_metadata_norm.loc[
            convertir_a_booleano_gspan(
                df_metadata_norm["belongs_serverless"]
            ),
            "architecture_normalized_gspan"
        ]
    ),

    "None": set(
        df_metadata_norm.loc[
            convertir_a_booleano_gspan(
                df_metadata_norm[
                    "belongs_none_analytical"
                ]
            ),
            "architecture_normalized_gspan"
        ]
    ),

    # Corresponde a tu categoría Edge no CloudFront.
    "Strictly Edge": set(
        df_metadata_norm.loc[
            convertir_a_booleano_gspan(
                df_metadata_norm[
                    "belongs_edge_no_cloudfront"
                ]
            ),
            "architecture_normalized_gspan"
        ]
    ),

    "FaaS": set(
        df_metadata_norm.loc[
            convertir_a_booleano_gspan(
                df_metadata_norm[
                    "belongs_serverless_faas"
                ]
            ),
            "architecture_normalized_gspan"
        ]
    )
}


print("Arquitecturas por categoría:\n")

for category_norm, architectures_norm in (
    category_architectures_norm.items()
):
    print(
        f"{category_norm:15s}: "
        f"{len(architectures_norm)}"
    )

Arquitecturas por categoría:

HPC            : 15
Edge           : 106
Serverless     : 352
None           : 40
Strictly Edge  : 77
FaaS           : 224


In [42]:
# ==========================================================
# VALIDAR RELACIÓN METADATA <-> GRAPHML
# ==========================================================

graph_stems_norm = {
    normalizar_nombre_arquitectura_gspan(
        file_path_norm
    )
    for file_path_norm in files_norm
}

validation_rows_norm = []

for category_norm, architectures_norm in (
    category_architectures_norm.items()
):

    matched_architectures_norm = (
        architectures_norm
        & graph_stems_norm
    )

    missing_graphml_norm = (
        architectures_norm
        - graph_stems_norm
    )

    validation_rows_norm.append({
        "category": category_norm,
        "architectures_metadata": len(
            architectures_norm
        ),
        "matched_graphml": len(
            matched_architectures_norm
        ),
        "without_graphml": len(
            missing_graphml_norm
        ),
        "missing_examples": sorted(
            missing_graphml_norm
        )[:5]
    })


df_category_validation_norm = pd.DataFrame(
    validation_rows_norm
)

display(df_category_validation_norm)

,category,architectures_metadata,matched_graphml,without_graphml,missing_examples
0,HPC,15,15,0,[]
1,Edge,106,106,0,[]
2,Serverless,352,352,0,[]
3,None,40,40,0,[]
4,Strictly Edge,77,77,0,[]
5,FaaS,224,224,0,[]


In [43]:
# ==========================================================
# EXPORTAR GRAPHML NORMALIZADOS A GSPAN POR CATEGORÍA
# ==========================================================

CATEGORY_GSPAN_DIR_NORM = os.path.join(
    OUTPUT_DIR_NORM,
    "gspan_input_files"
)

os.makedirs(
    CATEGORY_GSPAN_DIR_NORM,
    exist_ok=True
)


def nombre_seguro_categoria_gspan(category_name):
    return re.sub(
        r"[^a-z0-9]+",
        "_",
        category_name.casefold()
    ).strip("_")


def crear_archivo_gspan_categoria_norm(
    category_name,
    architecture_names,
    graph_files,
    service_catalog,
    edge_catalog
):
    """
    Crea un archivo gSpan para una categoría.

    Usa etiquetas normalizadas de servicios y mantiene
    un archivo JSON que relaciona graph_id con GraphML.
    """

    architecture_names_norm = {
        normalizar_nombre_arquitectura_gspan(
            architecture_name
        )
        for architecture_name in architecture_names
    }

    category_slug_norm = (
        nombre_seguro_categoria_gspan(
            category_name
        )
    )

    output_path_norm = os.path.join(
        CATEGORY_GSPAN_DIR_NORM,
        f"{category_slug_norm}_normalized.data"
    )

    mapping_path_norm = os.path.join(
        CATEGORY_GSPAN_DIR_NORM,
        f"{category_slug_norm}_graph_names.json"
    )

    graph_id_to_name_norm = {}
    category_graph_id_norm = 0

    with open(
        output_path_norm,
        "w",
        encoding="utf-8"
    ) as output_norm:

        for graph_file_norm in graph_files:

            graph_name_norm = os.path.basename(
                graph_file_norm
            )

            architecture_id_norm = (
                normalizar_nombre_arquitectura_gspan(
                    graph_name_norm
                )
            )

            # El GraphML no pertenece a la categoría.
            if architecture_id_norm not in (
                architecture_names_norm
            ):
                continue

            G_category_norm = nx.read_graphml(
                graph_file_norm
            )

            graph_id_to_name_norm[
                category_graph_id_norm
            ] = graph_name_norm

            output_norm.write(
                f"t # {category_graph_id_norm}\n"
            )

            node_to_local_id_norm = {
                node: local_id
                for local_id, node in enumerate(
                    G_category_norm.nodes()
                )
            }

            # Nodos con etiqueta de servicio normalizada.
            for node_norm, attrs_norm in (
                G_category_norm.nodes(data=True)
            ):

                local_id_norm = node_to_local_id_norm[
                    node_norm
                ]

                service_norm = normalizar_servicio_gspan(
                    attrs_norm.get("service", "")
                )

                service_id_norm = service_catalog[
                    service_norm
                ]

                output_norm.write(
                    f"v {local_id_norm} "
                    f"{service_id_norm}\n"
                )

            # Aristas dirigidas.
            for (
                source_norm,
                target_norm,
                attrs_norm
            ) in G_category_norm.edges(data=True):

                source_id_norm = node_to_local_id_norm[
                    source_norm
                ]

                target_id_norm = node_to_local_id_norm[
                    target_norm
                ]

                edge_type_norm = (
                    normalizar_tipo_arista_gspan(
                        attrs_norm.get("type", "")
                    )
                )

                edge_type_id_norm = edge_catalog[
                    edge_type_norm
                ]

                output_norm.write(
                    f"e {source_id_norm} "
                    f"{target_id_norm} "
                    f"{edge_type_id_norm}\n"
                )

            category_graph_id_norm += 1

        output_norm.write("t # -1\n")

    with open(
        mapping_path_norm,
        "w",
        encoding="utf-8"
    ) as mapping_file_norm:

        json.dump(
            graph_id_to_name_norm,
            mapping_file_norm,
            ensure_ascii=False,
            indent=2
        )

    return {
        "category": category_name,
        "gspan_file": output_path_norm,
        "mapping_file": mapping_path_norm,
        "num_graphs": category_graph_id_norm,
        "graph_id_to_name": graph_id_to_name_norm
    }

In [44]:
# ==========================================================
# CREAR UN ARCHIVO GSPAN POR CATEGORÍA
# ==========================================================

category_gspan_info_norm = {}

for category_name_norm, architecture_set_norm in (
    category_architectures_norm.items()
):

    category_info_norm = (
        crear_archivo_gspan_categoria_norm(
            category_name=category_name_norm,
            architecture_names=architecture_set_norm,
            graph_files=files_norm,
            service_catalog=global_service_to_id_norm,
            edge_catalog=global_edge_type_to_id_norm
        )
    )

    category_gspan_info_norm[
        category_name_norm
    ] = category_info_norm

    print(
        f"{category_name_norm:15s}: "
        f"{category_info_norm['num_graphs']:3d} grafos -> "
        f"{os.path.basename(category_info_norm['gspan_file'])}"
    )

HPC            :  15 grafos -> hpc_normalized.data
Edge           : 106 grafos -> edge_normalized.data
Serverless     : 352 grafos -> serverless_normalized.data
None           :  40 grafos -> none_normalized.data
Strictly Edge  :  77 grafos -> strictly_edge_normalized.data
FaaS           : 224 grafos -> faas_normalized.data


In [45]:
# ==========================================================
# PARSEAR RESULTADOS GSPAN NORMALIZADOS
# ==========================================================

def obtener_etiqueta_gspan_norm(catalogo, clave):
    """
    Permite buscar una clave tanto como entero como string.
    """

    return catalogo.get(
        clave,
        catalogo.get(
            str(clave),
            f"UNKNOWN_{clave}"
        )
    )


def parsear_salida_gspan_norm(
    texto,
    catalogo_nodos,
    catalogo_aristas,
    catalogo_grafos,
    total_grafos
):
    """
    Convierte la salida de gSpan en un DataFrame legible.
    """

    patrones_norm = []
    patron_actual_norm = None

    for linea_original_norm in texto.splitlines():

        linea_norm = linea_original_norm.strip()

        if not linea_norm:
            continue

        # Inicio de patrón.
        if linea_norm.startswith("t #"):

            if patron_actual_norm is not None:
                patrones_norm.append(
                    patron_actual_norm
                )

            partes_norm = linea_norm.split()

            pattern_id_norm = int(
                partes_norm[-1]
            )

            # Protección por si aparece una marca final t # -1.
            if pattern_id_norm < 0:
                patron_actual_norm = None
                continue

            patron_actual_norm = {
                "pattern_id": pattern_id_norm,
                "nodes": {},
                "edges": [],
                "support": None,
                "graph_ids": []
            }

        # Nodo: v node_id label_id.
        elif (
            linea_norm.startswith("v ")
            and patron_actual_norm is not None
        ):

            partes_norm = linea_norm.split()

            node_id_norm = int(partes_norm[1])
            label_id_norm = int(partes_norm[2])

            patron_actual_norm["nodes"][
                node_id_norm
            ] = {
                "label_id": label_id_norm,
                "label": obtener_etiqueta_gspan_norm(
                    catalogo_nodos,
                    label_id_norm
                )
            }

        # Arista: e source target edge_label_id.
        elif (
            linea_norm.startswith("e ")
            and patron_actual_norm is not None
        ):

            partes_norm = linea_norm.split()

            source_norm = int(partes_norm[1])
            target_norm = int(partes_norm[2])
            edge_label_id_norm = int(partes_norm[3])

            patron_actual_norm["edges"].append({
                "source": source_norm,
                "target": target_norm,
                "label_id": edge_label_id_norm,
                "label": obtener_etiqueta_gspan_norm(
                    catalogo_aristas,
                    edge_label_id_norm
                )
            })

        # Soporte.
        elif (
            linea_norm.startswith("Support:")
            and patron_actual_norm is not None
        ):

            patron_actual_norm["support"] = int(
                linea_norm.split(
                    ":",
                    1
                )[1].strip()
            )

        # Arquitecturas donde aparece el patrón.
        elif (
            linea_norm.startswith("where:")
            and patron_actual_norm is not None
        ):

            patron_actual_norm["graph_ids"] = [
                int(value_norm)
                for value_norm in re.findall(
                    r"\d+",
                    linea_norm
                )
            ]

    if patron_actual_norm is not None:
        patrones_norm.append(patron_actual_norm)

    rows_norm = []

    for patron_norm in patrones_norm:

        connections_norm = []

        for edge_norm in patron_norm["edges"]:

            source_label_norm = (
                patron_norm["nodes"][
                    edge_norm["source"]
                ]["label"]
            )

            target_label_norm = (
                patron_norm["nodes"][
                    edge_norm["target"]
                ]["label"]
            )

            connections_norm.append(
                f"{source_label_norm} "
                f"-[{edge_norm['label']}]-> "
                f"{target_label_norm}"
            )

        graph_ids_norm = patron_norm["graph_ids"]

        architecture_names_norm = [
            obtener_etiqueta_gspan_norm(
                catalogo_grafos,
                graph_id_norm
            )
            for graph_id_norm in graph_ids_norm
        ]

        support_norm = patron_norm["support"]

        rows_norm.append({
            "pattern_id": patron_norm["pattern_id"],
            "pattern": " | ".join(
                connections_norm
            ),
            "num_nodes": len(
                patron_norm["nodes"]
            ),
            "num_edges": len(
                patron_norm["edges"]
            ),
            "support": support_norm,
            "percentage_graphs": (
                100 * support_norm / total_grafos
                if support_norm is not None
                and total_grafos > 0
                else None
            ),
            "graph_ids": graph_ids_norm,
            "architectures": architecture_names_norm
        })

    df_patterns_norm = pd.DataFrame(rows_norm)

    if not df_patterns_norm.empty:

        df_patterns_norm = (
            df_patterns_norm
            .sort_values(
                by=[
                    "num_edges",
                    "support",
                    "pattern"
                ],
                ascending=[
                    True,
                    False,
                    True
                ]
            )
            .reset_index(drop=True)
        )

    return df_patterns_norm

###Limits


SUPPORT_RATIOS_NORM = [0.1, 0.06, 0.03]

MIN_NODES_NORM = 2
MAX_NODES_NORM = 4

10% of the graphs in the category

6% of the graphs in the category

3% of the graphs in the category

In [46]:
# ==========================================================
# EJECUTAR GSPAN POR CATEGORÍA
# ==========================================================

SUPPORT_RATIOS_NORM = [0.1, 0.06, 0.03]

MIN_NODES_NORM = 2
MAX_NODES_NORM = 4

RAW_RESULTS_DIR_NORM = os.path.join(
    OUTPUT_DIR_NORM,
    "raw_gspan_outputs"
)

CSV_RESULTS_DIR_NORM = os.path.join(
    OUTPUT_DIR_NORM,
    "parsed_results"
)

os.makedirs(
    RAW_RESULTS_DIR_NORM,
    exist_ok=True
)

os.makedirs(
    CSV_RESULTS_DIR_NORM,
    exist_ok=True
)


def ejecutar_gspan_categoria_norm(
    category_name,
    category_info,
    support_ratio
):
    """
    Ejecuta gSpan sobre una categoría y devuelve
    un DataFrame con patrones legibles.
    """

    total_graphs_norm = category_info["num_graphs"]

    if total_graphs_norm == 0:
        print(
            f"{category_name}: sin grafos; omitido."
        )
        return pd.DataFrame()

    minimum_support_norm = max(
        2,
        math.ceil(
            total_graphs_norm * support_ratio
        )
    )

    category_slug_norm = (
        nombre_seguro_categoria_gspan(
            category_name
        )
    )

    support_label_norm = (
        f"{int(support_ratio * 100):02d}"
    )

    raw_output_file_norm = os.path.join(
        RAW_RESULTS_DIR_NORM,
        f"{category_slug_norm}_"
        f"support_{support_label_norm}.txt"
    )

    command_norm = [
        "python",
        "-m",
        "gspan_mining",
        "-s", str(minimum_support_norm),
        "-l", str(MIN_NODES_NORM),
        "-u", str(MAX_NODES_NORM),
        "-d", "True",
        "-w", "True",
        category_info["gspan_file"]
    ]

    result_norm = subprocess.run(
        command_norm,
        capture_output=True,
        text=True
    )

    with open(
        raw_output_file_norm,
        "w",
        encoding="utf-8"
    ) as raw_file_norm:

        raw_file_norm.write(
            result_norm.stdout
        )

    if result_norm.returncode != 0:

        print(
            f"\nERROR en {category_name} "
            f"con soporte {support_ratio:.0%}:"
        )

        print(result_norm.stderr[:3000])

        return pd.DataFrame()

    df_category_patterns_norm = (
        parsear_salida_gspan_norm(
            texto=result_norm.stdout,
            catalogo_nodos=global_id_to_service_norm,
            catalogo_aristas=global_id_to_edge_type_norm,
            catalogo_grafos=category_info[
                "graph_id_to_name"
            ],
            total_grafos=total_graphs_norm
        )
    )

    if df_category_patterns_norm.empty:

        print(
            f"{category_name:15s} | "
            f"grafos={total_graphs_norm:3d} | "
            f"soporte={minimum_support_norm:3d} "
            f"({support_ratio:.0%}) | "
            f"sin patrones"
        )

        return df_category_patterns_norm

    df_category_patterns_norm.insert(
        0,
        "category",
        category_name
    )

    df_category_patterns_norm.insert(
        1,
        "support_ratio",
        support_ratio
    )

    df_category_patterns_norm.insert(
        2,
        "minimum_support",
        minimum_support_norm
    )

    df_category_patterns_norm.insert(
        3,
        "category_graphs",
        total_graphs_norm
    )

    csv_file_norm = os.path.join(
        CSV_RESULTS_DIR_NORM,
        f"{category_slug_norm}_"
        f"support_{support_label_norm}.csv"
    )

    df_category_patterns_norm.to_csv(
        csv_file_norm,
        index=False,
        encoding="utf-8-sig"
    )

    print(
        f"{category_name:15s} | "
        f"grafos={total_graphs_norm:3d} | "
        f"soporte={minimum_support_norm:3d} "
        f"({support_ratio:.0%}) | "
        f"patrones={len(df_category_patterns_norm):4d}"
    )

    return df_category_patterns_norm

In [47]:
# ==========================================================
# gspan_mining DEVUELVE CÓDIGO 1 AUNQUE TERMINE
# Esta celda no modifica las funciones anteriores.
# ==========================================================

def salida_gspan_valida_norm(result):
    """
    Detecta el comportamiento particular de gspan_mining:

    - Ejecuta correctamente la minería.
    - Escribe los patrones en stdout.
    - Pero finaliza con código 1 porque su __main__.py hace:
          sys.exit(main())
      y main() retorna un objeto gSpan.

    Por eso stderr contiene algo como:
        <gspan_mining.gspan.gSpan object at 0x...>
    """

    stderr_limpio = result.stderr.strip()

    patron_salida_gspan = re.fullmatch(
        r"<gspan_mining\.gspan\.gSpan object at 0x[0-9a-fA-F]+>",
        stderr_limpio
    )

    if result.returncode == 0:
        return True

    if (
        result.returncode == 1
        and patron_salida_gspan is not None
    ):
        return True

    return False


def ejecutar_gspan_categoria_norm_v2(
    category_name,
    category_info,
    support_ratio
):
    """
    Versión corregida de la ejecución por categoría.

    Acepta el código de retorno 1 específico de gspan_mining
    cuando stderr contiene solamente la representación del objeto
    gSpan y stdout contiene los resultados de la minería.
    """

    total_graphs_norm_v2 = category_info["num_graphs"]

    if total_graphs_norm_v2 == 0:
        print(
            f"{category_name}: sin grafos; omitido."
        )
        return pd.DataFrame()

    minimum_support_norm_v2 = max(
        2,
        math.ceil(
            total_graphs_norm_v2 * support_ratio
        )
    )

    category_slug_norm_v2 = (
        nombre_seguro_categoria_gspan(
            category_name
        )
    )

    support_label_norm_v2 = (
        f"{int(support_ratio * 100):02d}"
    )

    raw_output_file_norm_v2 = os.path.join(
        RAW_RESULTS_DIR_NORM,
        f"{category_slug_norm_v2}_"
        f"support_{support_label_norm_v2}_v2.txt"
    )

    command_norm_v2 = [
        "python",
        "-m",
        "gspan_mining",
        "-s", str(minimum_support_norm_v2),
        "-l", str(MIN_NODES_NORM),
        "-u", str(MAX_NODES_NORM),
        "-d", "True",
        "-w", "True",
        category_info["gspan_file"]
    ]

    result_norm_v2 = subprocess.run(
        command_norm_v2,
        capture_output=True,
        text=True
    )

    # Guardar siempre la salida, aun si ocurre un error real.
    with open(
        raw_output_file_norm_v2,
        "w",
        encoding="utf-8"
    ) as raw_file_norm_v2:

        raw_file_norm_v2.write(
            result_norm_v2.stdout
        )

    ejecucion_valida_norm_v2 = salida_gspan_valida_norm(
        result_norm_v2
    )

    if not ejecucion_valida_norm_v2:

        print(
            f"\nERROR REAL en {category_name} "
            f"con soporte {support_ratio:.0%}"
        )

        print(
            "\nCódigo de retorno:",
            result_norm_v2.returncode
        )

        print("\nSTDERR:")
        print(result_norm_v2.stderr[:3000])

        print("\nPrimeras líneas de STDOUT:")
        print(
            "\n".join(
                result_norm_v2.stdout
                .splitlines()[:30]
            )
        )

        return pd.DataFrame()

    # Parsear stdout incluso si returncode == 1,
    # porque corresponde al comportamiento normal del paquete.
    df_category_patterns_norm_v2 = (
        parsear_salida_gspan_norm(
            texto=result_norm_v2.stdout,
            catalogo_nodos=global_id_to_service_norm,
            catalogo_aristas=global_id_to_edge_type_norm,
            catalogo_grafos=category_info[
                "graph_id_to_name"
            ],
            total_grafos=total_graphs_norm_v2
        )
    )

    if df_category_patterns_norm_v2.empty:

        print(
            f"{category_name:15s} | "
            f"grafos={total_graphs_norm_v2:3d} | "
            f"soporte={minimum_support_norm_v2:3d} "
            f"({support_ratio:.0%}) | "
            f"ejecutado, pero sin patrones parseados"
        )

        return df_category_patterns_norm_v2

    df_category_patterns_norm_v2.insert(
        0,
        "category",
        category_name
    )

    df_category_patterns_norm_v2.insert(
        1,
        "support_ratio",
        support_ratio
    )

    df_category_patterns_norm_v2.insert(
        2,
        "minimum_support",
        minimum_support_norm_v2
    )

    df_category_patterns_norm_v2.insert(
        3,
        "category_graphs",
        total_graphs_norm_v2
    )

    csv_file_norm_v2 = os.path.join(
        CSV_RESULTS_DIR_NORM,
        f"{category_slug_norm_v2}_"
        f"support_{support_label_norm_v2}_v2.csv"
    )

    df_category_patterns_norm_v2.to_csv(
        csv_file_norm_v2,
        index=False,
        encoding="utf-8-sig"
    )

    print(
        f"{category_name:15s} | "
        f"grafos={total_graphs_norm_v2:3d} | "
        f"soporte={minimum_support_norm_v2:3d} "
        f"({support_ratio:.0%}) | "
        f"patrones={len(df_category_patterns_norm_v2):4d}"
    )

    return df_category_patterns_norm_v2# ==========================================================
# CORRECCIÓN: gspan_mining DEVUELVE CÓDIGO 1 AUNQUE TERMINE
# Esta celda no modifica las funciones anteriores.
# ==========================================================

def salida_gspan_valida_norm(result):
    """
    Detecta el comportamiento particular de gspan_mining:

    - Ejecuta correctamente la minería.
    - Escribe los patrones en stdout.
    - Pero finaliza con código 1 porque su __main__.py hace:
          sys.exit(main())
      y main() retorna un objeto gSpan.

    Por eso stderr contiene algo como:
        <gspan_mining.gspan.gSpan object at 0x...>
    """

    stderr_limpio = result.stderr.strip()

    patron_salida_gspan = re.fullmatch(
        r"<gspan_mining\.gspan\.gSpan object at 0x[0-9a-fA-F]+>",
        stderr_limpio
    )

    if result.returncode == 0:
        return True

    if (
        result.returncode == 1
        and patron_salida_gspan is not None
    ):
        return True

    return False


def ejecutar_gspan_categoria_norm_v2(
    category_name,
    category_info,
    support_ratio
):
    """
    Versión corregida de la ejecución por categoría.

    Acepta el código de retorno 1 específico de gspan_mining
    cuando stderr contiene solamente la representación del objeto
    gSpan y stdout contiene los resultados de la minería.
    """

    total_graphs_norm_v2 = category_info["num_graphs"]

    if total_graphs_norm_v2 == 0:
        print(
            f"{category_name}: sin grafos; omitido."
        )
        return pd.DataFrame()

    minimum_support_norm_v2 = max(
        2,
        math.ceil(
            total_graphs_norm_v2 * support_ratio
        )
    )

    category_slug_norm_v2 = (
        nombre_seguro_categoria_gspan(
            category_name
        )
    )

    support_label_norm_v2 = (
        f"{int(support_ratio * 100):02d}"
    )

    raw_output_file_norm_v2 = os.path.join(
        RAW_RESULTS_DIR_NORM,
        f"{category_slug_norm_v2}_"
        f"support_{support_label_norm_v2}_v2.txt"
    )

    command_norm_v2 = [
        "python",
        "-m",
        "gspan_mining",
        "-s", str(minimum_support_norm_v2),
        "-l", str(MIN_NODES_NORM),
        "-u", str(MAX_NODES_NORM),
        "-d", "True",
        "-w", "True",
        category_info["gspan_file"]
    ]

    result_norm_v2 = subprocess.run(
        command_norm_v2,
        capture_output=True,
        text=True
    )

    # Guardar siempre la salida, aun si ocurre un error real.
    with open(
        raw_output_file_norm_v2,
        "w",
        encoding="utf-8"
    ) as raw_file_norm_v2:

        raw_file_norm_v2.write(
            result_norm_v2.stdout
        )

    ejecucion_valida_norm_v2 = salida_gspan_valida_norm(
        result_norm_v2
    )

    if not ejecucion_valida_norm_v2:

        print(
            f"\nERROR REAL en {category_name} "
            f"con soporte {support_ratio:.0%}"
        )

        print(
            "\nCódigo de retorno:",
            result_norm_v2.returncode
        )

        print("\nSTDERR:")
        print(result_norm_v2.stderr[:3000])

        print("\nPrimeras líneas de STDOUT:")
        print(
            "\n".join(
                result_norm_v2.stdout
                .splitlines()[:30]
            )
        )

        return pd.DataFrame()

    # Parsear stdout incluso si returncode == 1,
    # porque corresponde al comportamiento normal del paquete.
    df_category_patterns_norm_v2 = (
        parsear_salida_gspan_norm(
            texto=result_norm_v2.stdout,
            catalogo_nodos=global_id_to_service_norm,
            catalogo_aristas=global_id_to_edge_type_norm,
            catalogo_grafos=category_info[
                "graph_id_to_name"
            ],
            total_grafos=total_graphs_norm_v2
        )
    )

    if df_category_patterns_norm_v2.empty:

        print(
            f"{category_name:15s} | "
            f"grafos={total_graphs_norm_v2:3d} | "
            f"soporte={minimum_support_norm_v2:3d} "
            f"({support_ratio:.0%}) | "
            f"ejecutado, pero sin patrones parseados"
        )

        return df_category_patterns_norm_v2

    df_category_patterns_norm_v2.insert(
        0,
        "category",
        category_name
    )

    df_category_patterns_norm_v2.insert(
        1,
        "support_ratio",
        support_ratio
    )

    df_category_patterns_norm_v2.insert(
        2,
        "minimum_support",
        minimum_support_norm_v2
    )

    df_category_patterns_norm_v2.insert(
        3,
        "category_graphs",
        total_graphs_norm_v2
    )

    csv_file_norm_v2 = os.path.join(
        CSV_RESULTS_DIR_NORM,
        f"{category_slug_norm_v2}_"
        f"support_{support_label_norm_v2}_v2.csv"
    )

    df_category_patterns_norm_v2.to_csv(
        csv_file_norm_v2,
        index=False,
        encoding="utf-8-sig"
    )

    print(
        f"{category_name:15s} | "
        f"grafos={total_graphs_norm_v2:3d} | "
        f"soporte={minimum_support_norm_v2:3d} "
        f"({support_ratio:.0%}) | "
        f"patrones={len(df_category_patterns_norm_v2):4d}"
    )

    return df_category_patterns_norm_v2

##All patterns normalized by category

In [48]:
# ==========================================================
# MINERÍA NORMALIZADA POR CATEGORÍA
# ==========================================================

all_category_patterns_norm_v2 = []

for category_name_norm_v2, category_info_norm_v2 in (
    category_gspan_info_norm.items()
):

    for support_ratio_norm_v2 in SUPPORT_RATIOS_NORM:

        df_category_patterns_norm_v2 = (
            ejecutar_gspan_categoria_norm_v2(
                category_name=category_name_norm_v2,
                category_info=category_info_norm_v2,
                support_ratio=support_ratio_norm_v2
            )
        )

        if not df_category_patterns_norm_v2.empty:

            all_category_patterns_norm_v2.append(
                df_category_patterns_norm_v2
            )


if all_category_patterns_norm_v2:

    df_patterns_by_category_norm_v2 = pd.concat(
        all_category_patterns_norm_v2,
        ignore_index=True
    )

    df_patterns_by_category_norm_v2 = (
        df_patterns_by_category_norm_v2
        .sort_values(
            by=[
                "category",
                "support_ratio",
                "num_edges",
                "percentage_graphs",
                "support"
            ],
            ascending=[
                True,
                False,
                False,
                False,
                False
            ]
        )
        .reset_index(drop=True)
    )

    consolidated_csv_norm_v2 = os.path.join(
        CSV_RESULTS_DIR_NORM,
        "all_patterns_normalized_by_category_v2.csv"
    )

    df_patterns_by_category_norm_v2.to_csv(
        consolidated_csv_norm_v2,
        index=False,
        encoding="utf-8-sig"
    )

    print("\nArchivo consolidado creado:")
    print(consolidated_csv_norm_v2)

    display(
        df_patterns_by_category_norm_v2[
            [
                "category",
                "support_ratio",
                "pattern",
                "num_nodes",
                "num_edges",
                "support",
                "percentage_graphs"
            ]
        ].head(150)
    )

else:

    df_patterns_by_category_norm_v2 = pd.DataFrame()

    print(
        "La ejecución terminó, pero no se pudieron "
        "parsear patrones desde stdout."
    )

HPC             | grafos= 15 | soporte=  2 (10%) | patrones=  23
HPC             | grafos= 15 | soporte=  2 (6%) | patrones=  23
HPC             | grafos= 15 | soporte=  2 (3%) | patrones=  23
Edge            | grafos=106 | soporte= 11 (10%) | patrones=  10
Edge            | grafos=106 | soporte=  7 (6%) | patrones=  24
Edge            | grafos=106 | soporte=  4 (3%) | patrones=  96
Serverless      | grafos=352 | soporte= 36 (10%) | patrones=   6
Serverless      | grafos=352 | soporte= 22 (6%) | patrones=  12
Serverless      | grafos=352 | soporte= 11 (3%) | patrones=  77
None            | grafos= 40 | soporte=  4 (10%) | patrones=   3
None            | grafos= 40 | soporte=  3 (6%) | patrones=   8
None            | grafos= 40 | soporte=  2 (3%) | patrones=  46
Strictly Edge   | grafos= 77 | soporte=  8 (10%) | patrones=  12
Strictly Edge   | grafos= 77 | soporte=  5 (6%) | patrones=  35
Strictly Edge   | grafos= 77 | soporte=  3 (3%) | patrones= 101
FaaS            | grafos=224 | sopo

,category,support_ratio,pattern,num_nodes,num_edges,support,percentage_graphs
0,Edge,0.10,UserConsumer -[data]-> CloudFront,2,1,19,17.924528
1,Edge,0.10,CloudFront -[data]-> UserConsumer,2,1,15,14.150943
2,Edge,0.10,Lambda -[data]-> DynamoDB,2,1,15,14.150943
3,Edge,0.10,S3 -[data]-> CloudFront,2,1,15,14.150943
4,Edge,0.10,CloudFront -[data]-> S3,2,1,13,12.264151
...,...,...,...,...,...,...,...
145,FaaS,0.06,Lambda -[data]-> S3,2,1,43,19.196429
146,FaaS,0.06,S3 -[data]-> Lambda,2,1,40,17.857143
147,FaaS,0.06,DynamoDB -[data]-> Lambda,2,1,37,16.517857
148,FaaS,0.06,UserConsumer -[data]-> ApiGateway,2,1,27,12.053571


In [49]:
df_structural_patterns_norm_v2 = (
    df_patterns_by_category_norm_v2[
        df_patterns_by_category_norm_v2["num_edges"] >= 2
    ]
    .copy()
)

display(
    df_structural_patterns_norm_v2[
        [
            "category",
            "support_ratio",
            "pattern",
            "num_nodes",
            "num_edges",
            "support",
            "percentage_graphs"
        ]
    ].head(200)
)

,category,support_ratio,pattern,num_nodes,num_edges,support,percentage_graphs
10,Edge,0.06,S3 -[data]-> CloudFront | CloudFront -[meta]-> S3,2,2,10,9.433962
11,Edge,0.06,S3 -[data]-> CloudFront | CloudFront -[data]->...,3,2,8,7.547170
12,Edge,0.06,UserConsumer -[data]-> CloudFront | CloudFront...,3,2,8,7.547170
34,Edge,0.03,DynamoDB -[data]-> Lambda | Lambda -[meta]-> D...,3,4,4,3.773585
35,Edge,0.03,CloudFront -[data]-> S3 | CloudFront -[data]->...,3,3,4,3.773585
...,...,...,...,...,...,...,...
551,Strictly Edge,0.03,S3 -[data]-> Lambda | Lambda -[meta]-> S3,2,2,3,3.896104
552,Strictly Edge,0.03,S3 -[meta]-> EC2 | EC2 -[data]-> S3,2,2,3,3.896104
553,Strictly Edge,0.03,StepFunctions -[data]-> Lambda | StepFunctions...,3,2,3,3.896104
554,Strictly Edge,0.03,UserConsumer -[data]-> ApiGateway | ApiGateway...,3,2,3,3.896104


# BOOTSTRAP VALIDATION

In [51]:
# ==========================================================
# BOOTSTRAP VALIDATION OF gSpan RESULTS
# ==========================================================

import os
import json
import math
import hashlib
import subprocess

from collections import Counter, defaultdict
from itertools import permutations

import numpy as np
import pandas as pd
import networkx as nx

from IPython.display import display


# ==========================================================
# COMPATIBILITY ALIASES FOR THE PREVIOUS GSPAN CELLS
# These aliases keep all NEW bootstrap code in English while
# using objects that were already created in previous cells.
# ==========================================================

required_previous_objects = [
    "files_norm",
    "category_architectures_norm",
    "category_gspan_info_norm",
    "global_service_to_id_norm",
    "global_edge_type_to_id_norm",
    "global_id_to_service_norm",
    "global_id_to_edge_type_norm",
    "normalizar_nombre_arquitectura_gspan",
    "normalizar_servicio_gspan",
    "normalizar_tipo_arista_gspan",
    "nombre_seguro_categoria_gspan",
    "parsear_salida_gspan_norm",
    "salida_gspan_valida_norm",
    "SUPPORT_RATIOS_NORM",
    "MIN_NODES_NORM",
    "MAX_NODES_NORM",
    "OUTPUT_DIR_NORM"
]

missing_previous_objects = [
    object_name
    for object_name in required_previous_objects
    if object_name not in globals()
]

if missing_previous_objects:
    raise RuntimeError(
        "Run all original gSpan cells before running this "
        "bootstrap validation cell.\n\n"
        f"Missing objects:\n{missing_previous_objects}"
    )

# English aliases for objects created in the original cells.
graph_files = files_norm
category_architectures = category_architectures_norm
category_gspan_info = category_gspan_info_norm

service_to_id = global_service_to_id_norm
edge_type_to_id = global_edge_type_to_id_norm

id_to_service = global_id_to_service_norm
id_to_edge_type = global_id_to_edge_type_norm

normalize_architecture_name = (
    normalizar_nombre_arquitectura_gspan
)

normalize_service = normalizar_servicio_gspan
normalize_edge_type = normalizar_tipo_arista_gspan

safe_category_name = nombre_seguro_categoria_gspan
parse_gspan_output = parsear_salida_gspan_norm
is_valid_gspan_result = salida_gspan_valida_norm

support_ratios = list(SUPPORT_RATIOS_NORM)
min_nodes = MIN_NODES_NORM
max_nodes = MAX_NODES_NORM
main_output_directory = OUTPUT_DIR_NORM


# ==========================================================
# CONFIGURATION
# ==========================================================

# Start with 10 iterations.
# Change only this value to 5000 after verifying that all
# tables and CSV files are generated correctly.
BOOTSTRAP_ITERATIONS = 10

# Fixed seed for reproducible bootstrap samples.
BOOTSTRAP_RANDOM_SEED = 20260704

# Do not run bootstrap for groups with fewer than 100 graphs.
BOOTSTRAP_MINIMUM_GRAPHS = 70

# Explicit exclusions requested for small groups.
BOOTSTRAP_EXCLUDED_CATEGORIES = {
    "HPC",
    "None"
}

# Use the same support ratios as the original gSpan analysis.
BOOTSTRAP_SUPPORT_RATIOS = support_ratios.copy()

# patterns stored in df_structural_patterns_norm_v2.
BOOTSTRAP_ONLY_STRUCTURAL_PATTERNS = False

# Do not change this to True for the 5000-iteration run.
# Otherwise, thousands of temporary .data files will remain.
BOOTSTRAP_KEEP_TEMPORARY_INPUTS = False


# ==========================================================
# INPUT VALIDATION
# ==========================================================

if BOOTSTRAP_ITERATIONS < 1:
    raise ValueError(
        "BOOTSTRAP_ITERATIONS must be greater than or equal to 1."
    )

if not BOOTSTRAP_SUPPORT_RATIOS:
    raise ValueError(
        "BOOTSTRAP_SUPPORT_RATIOS cannot be empty."
    )

invalid_support_ratios = [
    support_ratio
    for support_ratio in BOOTSTRAP_SUPPORT_RATIOS
    if support_ratio <= 0 or support_ratio > 1
]

if invalid_support_ratios:
    raise ValueError(
        "Every support ratio must be within the interval (0, 1].\n"
        f"Invalid values: {invalid_support_ratios}"
    )


# ==========================================================
# OUTPUT DIRECTORIES
# ==========================================================

bootstrap_directory = os.path.join(
    main_output_directory,
    "bootstrap_validation"
)

temporary_input_directory = os.path.join(
    bootstrap_directory,
    "temporary_inputs"
)

results_directory = os.path.join(
    bootstrap_directory,
    "results"
)

os.makedirs(
    temporary_input_directory,
    exist_ok=True
)

os.makedirs(
    results_directory,
    exist_ok=True
)


# ==========================================================
# GRAPHML FILE INDEX
# ==========================================================

graph_file_by_architecture = {}

for graph_file in graph_files:

    architecture_name = normalize_architecture_name(
        graph_file
    )

    if architecture_name in graph_file_by_architecture:
        raise ValueError(
            "Duplicate normalized architecture name found "
            "among GraphML files:\n"
            f"{architecture_name}"
        )

    graph_file_by_architecture[
        architecture_name
    ] = graph_file


# ==========================================================
# GRAPH CACHE
# ==========================================================

graph_cache = {}


def get_cached_graph(architecture_name):
    """
    Loads a GraphML graph only once and then keeps it in memory.

    This avoids repeatedly reading the same GraphML file during
    bootstrap iterations.
    """

    if architecture_name not in graph_file_by_architecture:
        raise KeyError(
            "No GraphML file was found for architecture:\n"
            f"{architecture_name}"
        )

    if architecture_name not in graph_cache:
        graph_cache[architecture_name] = nx.read_graphml(
            graph_file_by_architecture[
                architecture_name
            ]
        )

    return graph_cache[architecture_name]


# ==========================================================
# CANONICAL STRUCTURAL SIGNATURE FOR PATTERN COMPARISON
# ==========================================================

def create_canonical_pattern_signature(nodes, edges):
    """
    Creates a canonical structural signature for a gSpan pattern.

    The signature is independent of:
    - gSpan pattern IDs;
    - internal node numbering;
    - pattern discovery order.

    The signature preserves:
    - node labels;
    - directed edges;
    - edge labels;
    - graph structure.

    Since max_nodes is 4, testing all node permutations remains
    computationally manageable.
    """

    original_node_ids = sorted(nodes)

    if not original_node_ids:
        raise ValueError(
            "A pattern without nodes cannot have a signature."
        )

    candidate_signatures = []

    for ordered_node_ids in permutations(
        original_node_ids
    ):

        old_to_new_node_id = {
            old_node_id: new_node_id
            for new_node_id, old_node_id in enumerate(
                ordered_node_ids
            )
        }

        ordered_node_labels = tuple(
            int(
                nodes[old_node_id]["label_id"]
            )
            for old_node_id in ordered_node_ids
        )

        ordered_edges = tuple(
            sorted(
                (
                    old_to_new_node_id[
                        edge["source"]
                    ],
                    old_to_new_node_id[
                        edge["target"]
                    ],
                    int(edge["label_id"])
                )
                for edge in edges
            )
        )

        candidate_signatures.append(
            (
                ordered_node_labels,
                ordered_edges
            )
        )

    best_signature = min(
        candidate_signatures
    )

    return json.dumps(
        {
            "node_labels": list(
                best_signature[0]
            ),
            "edges": [
                list(edge)
                for edge in best_signature[1]
            ]
        },
        separators=(",", ":")
    )


def extract_pattern_signatures(gspan_stdout):
    """
    Extracts t/v/e pattern blocks from gSpan output and returns:

        pattern_id -> canonical structural signature
    """

    signatures = {}

    current_pattern = None

    def save_current_pattern():

        if current_pattern is None:
            return

        signatures[
            current_pattern["pattern_id"]
        ] = create_canonical_pattern_signature(
            current_pattern["nodes"],
            current_pattern["edges"]
        )

    for raw_line in gspan_stdout.splitlines():

        line = raw_line.strip()

        if not line:
            continue

        # Pattern beginning.
        if line.startswith("t #"):

            save_current_pattern()

            pattern_id = int(
                line.split()[-1]
            )

            if pattern_id < 0:
                current_pattern = None

            else:
                current_pattern = {
                    "pattern_id": pattern_id,
                    "nodes": {},
                    "edges": []
                }

        # Pattern node.
        elif (
            line.startswith("v ")
            and current_pattern is not None
        ):

            _, node_id, label_id = line.split()[:3]

            current_pattern["nodes"][
                int(node_id)
            ] = {
                "label_id": int(label_id)
            }

        # Pattern edge.
        elif (
            line.startswith("e ")
            and current_pattern is not None
        ):

            (
                _,
                source_node,
                target_node,
                edge_label_id
            ) = line.split()[:4]

            current_pattern["edges"].append({
                "source": int(source_node),
                "target": int(target_node),
                "label_id": int(edge_label_id)
            })

    save_current_pattern()

    return signatures


def create_pattern_key(pattern_signature):
    """
    Creates a short readable identifier for a structural signature.
    """

    return hashlib.sha1(
        pattern_signature.encode("utf-8")
    ).hexdigest()[:16]


# ==========================================================
# BOOTSTRAP GSPAN INPUT FILE CREATION
# ==========================================================

def write_bootstrap_gspan_input(
    category_name,
    sampled_architectures,
    iteration
):
    """
    Writes one bootstrap sample as a gSpan input file.

    IMPORTANT:
    sampled_architectures must be a list-like sequence, not a set.

    Bootstrap uses sampling with replacement. Therefore, the same
    architecture may appear multiple times. Each occurrence is written
    as an independent graph with a different graph ID.
    """

    if not isinstance(
        sampled_architectures,
        (
            list,
            tuple,
            np.ndarray
        )
    ):
        raise TypeError(
            "sampled_architectures must be a list, tuple, or array. "
            "Do not use a set because bootstrap duplicates must remain."
        )

    if len(sampled_architectures) == 0:
        raise ValueError(
            "The bootstrap sample cannot be empty."
        )

    category_slug = safe_category_name(
        category_name
    )

    output_file = os.path.join(
        temporary_input_directory,
        f"{category_slug}_"
        f"bootstrap_{iteration:05d}.data"
    )

    graph_id_to_architecture = {}

    with open(
        output_file,
        "w",
        encoding="utf-8"
    ) as output_file_handle:

        for graph_id, architecture_name in enumerate(
            sampled_architectures
        ):

            graph = get_cached_graph(
                architecture_name
            )

            # The same architecture can appear multiple times.
            # Each appearance receives a distinct graph ID.
            graph_id_to_architecture[
                graph_id
            ] = architecture_name

            output_file_handle.write(
                f"t # {graph_id}\n"
            )

            local_node_ids = {
                node: local_node_id
                for local_node_id, node in enumerate(
                    graph.nodes()
                )
            }

            # Write nodes.
            for node, attributes in graph.nodes(data=True):

                normalized_service = normalize_service(
                    attributes.get(
                        "service",
                        ""
                    )
                )

                service_id = service_to_id[
                    normalized_service
                ]

                output_file_handle.write(
                    f"v "
                    f"{local_node_ids[node]} "
                    f"{service_id}\n"
                )

            # Write directed edges.
            for (
                source_node,
                target_node,
                attributes
            ) in graph.edges(data=True):

                normalized_edge_type = normalize_edge_type(
                    attributes.get(
                        "type",
                        ""
                    )
                )

                edge_type_id = edge_type_to_id[
                    normalized_edge_type
                ]

                output_file_handle.write(
                    f"e "
                    f"{local_node_ids[source_node]} "
                    f"{local_node_ids[target_node]} "
                    f"{edge_type_id}\n"
                )

        output_file_handle.write(
            "t # -1\n"
        )

    return {
        "gspan_file": output_file,
        "graph_id_to_architecture": (
            graph_id_to_architecture
        ),
        "number_of_graphs": len(
            sampled_architectures
        )
    }


# ==========================================================
# RUN GSPAN AND ADD STRUCTURAL PATTERN SIGNATURES
# ==========================================================

def run_gspan_for_bootstrap(
    gspan_file,
    graph_id_to_architecture,
    number_of_graphs,
    support_ratio
):
    """
    Runs gSpan and returns a DataFrame with:

    - pattern_signature
    - pattern_key

    Returns:
        patterns_dataframe, None

    In case of a real execution error:
        empty_dataframe, error_message
    """

    minimum_support = max(
        2,
        math.ceil(
            number_of_graphs
            * support_ratio
        )
    )

    command = [
        "python",
        "-m",
        "gspan_mining",
        "-s",
        str(minimum_support),
        "-l",
        str(min_nodes),
        "-u",
        str(max_nodes),
        "-d",
        "True",
        "-w",
        "True",
        gspan_file
    ]

    result = subprocess.run(
        command,
        capture_output=True,
        text=True
    )

    if not is_valid_gspan_result(result):

        stdout_preview = "\n".join(
            result.stdout
            .splitlines()[:30]
        )

        error_message = (
            f"returncode={result.returncode}\n\n"
            f"stderr:\n{result.stderr[:3000]}\n\n"
            f"stdout preview:\n{stdout_preview}"
        )

        return pd.DataFrame(), error_message

    patterns_dataframe = parse_gspan_output(
        texto=result.stdout,
        catalogo_nodos=id_to_service,
        catalogo_aristas=id_to_edge_type,
        catalogo_grafos=graph_id_to_architecture,
        total_grafos=number_of_graphs
    )

    if patterns_dataframe.empty:
        return patterns_dataframe, None

    pattern_signatures = extract_pattern_signatures(
        result.stdout
    )

    patterns_dataframe = patterns_dataframe.copy()

    patterns_dataframe["pattern_signature"] = (
        patterns_dataframe["pattern_id"].map(
            pattern_signatures
        )
    )

    unresolved_pattern_ids = (
        patterns_dataframe.loc[
            patterns_dataframe[
                "pattern_signature"
            ].isna(),
            "pattern_id"
        ]
        .tolist()
    )

    if unresolved_pattern_ids:
        return (
            pd.DataFrame(),
            (
                "A canonical signature could not be generated for "
                "these pattern IDs:\n"
                f"{unresolved_pattern_ids}"
            )
        )

    patterns_dataframe["pattern_key"] = (
        patterns_dataframe[
            "pattern_signature"
        ].map(
            create_pattern_key
        )
    )

    # A structural signature should identify each graph pattern
    # uniquely. This removes any accidental duplicates safely.
    patterns_dataframe = (
        patterns_dataframe
        .drop_duplicates(
            subset=["pattern_signature"]
        )
        .reset_index(drop=True)
    )

    return patterns_dataframe, None


# ==========================================================
# BOOTSTRAP METRIC FUNCTIONS
# ==========================================================

def wilson_confidence_interval(
    successes,
    total,
    z_score=1.959963984540054
):
    """
    Calculates a 95% Wilson confidence interval for a proportion.
    """

    if total <= 0:
        return np.nan, np.nan

    proportion = successes / total
    z_squared = z_score ** 2

    denominator = (
        1
        + z_squared / total
    )

    center = (
        proportion
        + z_squared / (2 * total)
    ) / denominator

    half_width = (
        z_score
        * math.sqrt(
            (
                proportion
                * (1 - proportion)
                + z_squared / (4 * total)
            )
            / total
        )
        / denominator
    )

    lower_bound = max(
        0,
        center - half_width
    )

    upper_bound = min(
        1,
        center + half_width
    )

    return lower_bound, upper_bound


def classify_stability(bootstrap_frequency_percentage):
    """
    Assigns an interpretable stability class.
    """

    if pd.isna(
        bootstrap_frequency_percentage
    ):
        return "Not estimated"

    if bootstrap_frequency_percentage >= 95:
        return "High stability"

    if bootstrap_frequency_percentage >= 80:
        return "Moderate stability"

    return "Low stability"


# ==========================================================
# SELECT ELIGIBLE CATEGORIES
# ==========================================================

bootstrap_population_by_category = {}

category_selection_rows = []

for (
    category_name,
    category_architecture_names
) in category_architectures.items():

    available_architectures = sorted(
        set(category_architecture_names)
        & set(graph_file_by_architecture)
    )

    original_gspan_graph_count = (
        category_gspan_info[
            category_name
        ]["num_graphs"]
    )

    exclusion_reasons = []

    if category_name in BOOTSTRAP_EXCLUDED_CATEGORIES:
        exclusion_reasons.append(
            "Explicitly excluded"
        )

    if len(available_architectures) < BOOTSTRAP_MINIMUM_GRAPHS:
        exclusion_reasons.append(
            f"Fewer than "
            f"{BOOTSTRAP_MINIMUM_GRAPHS} graphs"
        )

    if len(available_architectures) != original_gspan_graph_count:
        exclusion_reasons.append(
            "Graph count differs from the original gSpan input"
        )

    is_selected = not exclusion_reasons

    category_selection_rows.append({
        "category": category_name,
        "available_graphs": len(
            available_architectures
        ),
        "original_gspan_graphs": (
            original_gspan_graph_count
        ),
        "selected_for_bootstrap": is_selected,
        "selection_reason": (
            "Eligible"
            if is_selected
            else "; ".join(
                exclusion_reasons
            )
        )
    })

    if is_selected:
        bootstrap_population_by_category[
            category_name
        ] = available_architectures


df_bootstrap_category_selection = (
    pd.DataFrame(
        category_selection_rows
    )
    .sort_values(
        "category"
    )
    .reset_index(drop=True)
)

print(
    "Bootstrap category selection:"
)

display(
    df_bootstrap_category_selection
)

if not bootstrap_population_by_category:
    raise RuntimeError(
        "No category is eligible for bootstrap validation with "
        "the current configuration."
    )


# ==========================================================
# RUN ORIGINAL BASELINE PATTERNS
# ==========================================================

baseline_patterns = {}

baseline_error_rows = []

for category_name in bootstrap_population_by_category:

    category_information = category_gspan_info[
        category_name
    ]

    for support_ratio in BOOTSTRAP_SUPPORT_RATIOS:

        (
            baseline_dataframe,
            baseline_error
        ) = run_gspan_for_bootstrap(
            gspan_file=category_information[
                "gspan_file"
            ],
            graph_id_to_architecture=category_information[
                "graph_id_to_name"
            ],
            number_of_graphs=category_information[
                "num_graphs"
            ],
            support_ratio=support_ratio
        )

        if baseline_error is not None:

            baseline_error_rows.append({
                "category": category_name,
                "support_ratio": support_ratio,
                "error": baseline_error
            })

            continue

        if (
            BOOTSTRAP_ONLY_STRUCTURAL_PATTERNS
            and not baseline_dataframe.empty
        ):
            baseline_dataframe = (
                baseline_dataframe.loc[
                    baseline_dataframe[
                        "num_edges"
                    ] >= 2
                ]
                .copy()
            )

        baseline_patterns[
            (
                category_name,
                support_ratio
            )
        ] = baseline_dataframe


if baseline_error_rows:

    print(
        "Errors occurred while generating the original baseline "
        "patterns. Bootstrap validation was not started."
    )

    display(
        pd.DataFrame(
            baseline_error_rows
        )
    )

    raise RuntimeError(
        "Baseline gSpan execution failed. Resolve the errors before "
        "running bootstrap validation."
    )


baseline_summary_rows = []

for (
    category_name,
    support_ratio
), baseline_dataframe in baseline_patterns.items():

    total_baseline_patterns = len(
        baseline_dataframe
    )

    structural_baseline_patterns = int(
        (
            baseline_dataframe[
                "num_edges"
            ] >= 2
        ).sum()
    ) if not baseline_dataframe.empty else 0

    baseline_summary_rows.append({
        "category": category_name,
        "support_ratio": support_ratio,
        "baseline_patterns": (
            total_baseline_patterns
        ),
        "baseline_structural_patterns": (
            structural_baseline_patterns
        )
    })


df_bootstrap_baseline_summary = (
    pd.DataFrame(
        baseline_summary_rows
    )
    .sort_values(
        [
            "category",
            "support_ratio"
        ]
    )
    .reset_index(drop=True)
)

print(
    "Original baseline patterns to validate:"
)

display(
    df_bootstrap_baseline_summary
)


# ==========================================================
# EXECUTE BOOTSTRAP ITERATIONS
# ==========================================================

random_generator = np.random.default_rng(
    BOOTSTRAP_RANDOM_SEED
)

# Number of bootstrap reproductions for every original pattern.
pattern_reproductions = Counter()

# Bootstrap support values observed when original patterns reappear.
recovered_support_values = defaultdict(list)

# Number of successful executions by category and support ratio.
successful_runs = Counter()

# General metrics for each bootstrap iteration.
iteration_metric_rows = []

# Real gSpan execution errors.
execution_error_rows = []


for (
    category_name,
    category_population
) in bootstrap_population_by_category.items():

    category_size = len(
        category_population
    )

    print(
        f"\nBootstrap validation for {category_name}: "
        f"{category_size} graphs, "
        f"{BOOTSTRAP_ITERATIONS} iterations"
    )

    progress_step = max(
        1,
        BOOTSTRAP_ITERATIONS // 10
    )

    for iteration in range(
        1,
        BOOTSTRAP_ITERATIONS + 1
    ):

        # Sample the same number of graphs with replacement.
        sampled_architectures = (
            random_generator.choice(
                category_population,
                size=category_size,
                replace=True
            )
            .tolist()
        )

        bootstrap_input_information = None

        try:

            bootstrap_input_information = (
                write_bootstrap_gspan_input(
                    category_name=category_name,
                    sampled_architectures=sampled_architectures,
                    iteration=iteration
                )
            )

            # Reuse the exact same sample for all support thresholds.
            for support_ratio in BOOTSTRAP_SUPPORT_RATIOS:

                baseline_dataframe = baseline_patterns[
                    (
                        category_name,
                        support_ratio
                    )
                ]

                (
                    bootstrap_dataframe,
                    execution_error
                ) = run_gspan_for_bootstrap(
                    gspan_file=bootstrap_input_information[
                        "gspan_file"
                    ],
                    graph_id_to_architecture=(
                        bootstrap_input_information[
                            "graph_id_to_architecture"
                        ]
                    ),
                    number_of_graphs=(
                        bootstrap_input_information[
                            "number_of_graphs"
                        ]
                    ),
                    support_ratio=support_ratio
                )

                if execution_error is not None:

                    execution_error_rows.append({
                        "category": category_name,
                        "iteration": iteration,
                        "support_ratio": support_ratio,
                        "error": execution_error
                    })

                    continue

                if (
                    BOOTSTRAP_ONLY_STRUCTURAL_PATTERNS
                    and not bootstrap_dataframe.empty
                ):
                    bootstrap_dataframe = (
                        bootstrap_dataframe.loc[
                            bootstrap_dataframe[
                                "num_edges"
                            ] >= 2
                        ]
                        .copy()
                    )

                successful_runs[
                    (
                        category_name,
                        support_ratio
                    )
                ] += 1

                # When baseline has no patterns, there are no
                # individual patterns to validate.
                if baseline_dataframe.empty:

                    iteration_metric_rows.append({
                        "category": category_name,
                        "iteration": iteration,
                        "support_ratio": support_ratio,
                        "original_pattern_count": 0,
                        "bootstrap_pattern_count": len(
                            bootstrap_dataframe
                        ),
                        "shared_pattern_count": 0,
                        "original_pattern_recall_pct": np.nan,
                        "bootstrap_pattern_precision_pct": np.nan,
                        "jaccard_similarity_pct": np.nan,
                        "exact_same_pattern_set": (
                            bootstrap_dataframe.empty
                        )
                    })

                    continue

                original_signatures = set(
                    baseline_dataframe[
                        "pattern_signature"
                    ]
                )

                bootstrap_signatures = set(
                    bootstrap_dataframe[
                        "pattern_signature"
                    ]
                ) if not bootstrap_dataframe.empty else set()

                shared_signatures = (
                    original_signatures
                    & bootstrap_signatures
                )

                union_signatures = (
                    original_signatures
                    | bootstrap_signatures
                )

                if bootstrap_dataframe.empty:
                    support_by_signature = {}

                else:
                    support_by_signature = (
                        bootstrap_dataframe
                        .groupby(
                            "pattern_signature"
                        )["support"]
                        .max()
                        .to_dict()
                    )

                # Count one reproduction for each original pattern
                # that appears again in the bootstrap sample.
                for pattern_signature in shared_signatures:

                    reproduction_key = (
                        category_name,
                        support_ratio,
                        pattern_signature
                    )

                    pattern_reproductions[
                        reproduction_key
                    ] += 1

                    recovered_support_values[
                        reproduction_key
                    ].append(
                        support_by_signature[
                            pattern_signature
                        ]
                    )

                original_pattern_count = len(
                    original_signatures
                )

                bootstrap_pattern_count = len(
                    bootstrap_signatures
                )

                shared_pattern_count = len(
                    shared_signatures
                )

                original_pattern_recall = (
                    shared_pattern_count
                    / original_pattern_count
                    if original_pattern_count > 0
                    else np.nan
                )

                bootstrap_pattern_precision = (
                    shared_pattern_count
                    / bootstrap_pattern_count
                    if bootstrap_pattern_count > 0
                    else np.nan
                )

                jaccard_similarity = (
                    shared_pattern_count
                    / len(union_signatures)
                    if union_signatures
                    else np.nan
                )

                iteration_metric_rows.append({
                    "category": category_name,
                    "iteration": iteration,
                    "support_ratio": support_ratio,
                    "original_pattern_count": (
                        original_pattern_count
                    ),
                    "bootstrap_pattern_count": (
                        bootstrap_pattern_count
                    ),
                    "shared_pattern_count": (
                        shared_pattern_count
                    ),
                    "original_pattern_recall_pct": (
                        100
                        * original_pattern_recall
                    ),
                    "bootstrap_pattern_precision_pct": (
                        100
                        * bootstrap_pattern_precision
                    ),
                    "jaccard_similarity_pct": (
                        100
                        * jaccard_similarity
                    ),
                    "exact_same_pattern_set": (
                        original_signatures
                        == bootstrap_signatures
                    )
                })

        finally:

            if (
                bootstrap_input_information is not None
                and not BOOTSTRAP_KEEP_TEMPORARY_INPUTS
            ):

                temporary_file = (
                    bootstrap_input_information[
                        "gspan_file"
                    ]
                )

                if os.path.exists(temporary_file):
                    os.remove(temporary_file)

        if (
            iteration == 1
            or iteration % progress_step == 0
            or iteration == BOOTSTRAP_ITERATIONS
        ):
            print(
                f"  Completed "
                f"{iteration}/{BOOTSTRAP_ITERATIONS} iterations"
            )


# ==========================================================
# PATTERN STABILITY RESULTS
# ==========================================================

pattern_stability_rows = []

for (
    category_name,
    support_ratio
), baseline_dataframe in baseline_patterns.items():

    successful_iterations = successful_runs[
        (
            category_name,
            support_ratio
        )
    ]

    if baseline_dataframe.empty:
        continue

    for _, original_pattern_row in (
        baseline_dataframe.iterrows()
    ):

        pattern_signature = original_pattern_row[
            "pattern_signature"
        ]

        reproduction_key = (
            category_name,
            support_ratio,
            pattern_signature
        )

        bootstrap_reproductions = pattern_reproductions[
            reproduction_key
        ]

        bootstrap_frequency = (
            bootstrap_reproductions
            / successful_iterations
            if successful_iterations > 0
            else np.nan
        )

        (
            confidence_interval_low,
            confidence_interval_high
        ) = wilson_confidence_interval(
            bootstrap_reproductions,
            successful_iterations
        )

        recovered_supports = recovered_support_values[
            reproduction_key
        ]

        category_graph_count = len(
            bootstrap_population_by_category[
                category_name
            ]
        )

        minimum_support = max(
            2,
            math.ceil(
                category_graph_count
                * support_ratio
            )
        )

        pattern_stability_rows.append({
            "category": category_name,
            "support_ratio": support_ratio,
            "minimum_support": minimum_support,
            "category_graphs": category_graph_count,
            "pattern_key": original_pattern_row[
                "pattern_key"
            ],
            "pattern": original_pattern_row[
                "pattern"
            ],
            "num_nodes": int(
                original_pattern_row[
                    "num_nodes"
                ]
            ),
            "num_edges": int(
                original_pattern_row[
                    "num_edges"
                ]
            ),
            "original_support": int(
                original_pattern_row[
                    "support"
                ]
            ),
            "original_percentage_graphs": float(
                original_pattern_row[
                    "percentage_graphs"
                ]
            ),
            "bootstrap_reproductions": (
                bootstrap_reproductions
            ),
            "successful_bootstrap_iterations": (
                successful_iterations
            ),
            "bootstrap_frequency_pct": (
                100
                * bootstrap_frequency
            ),
            "bootstrap_frequency_ci95_low_pct": (
                100
                * confidence_interval_low
            ),
            "bootstrap_frequency_ci95_high_pct": (
                100
                * confidence_interval_high
            ),
            "mean_support_when_recovered": (
                float(
                    np.mean(
                        recovered_supports
                    )
                )
                if recovered_supports
                else np.nan
            ),
            "stability_class": classify_stability(
                100
                * bootstrap_frequency
            )
        })


df_bootstrap_pattern_stability = pd.DataFrame(
    pattern_stability_rows
)

if not df_bootstrap_pattern_stability.empty:

    df_bootstrap_pattern_stability = (
        df_bootstrap_pattern_stability
        .sort_values(
            [
                "category",
                "support_ratio",
                "bootstrap_frequency_pct",
                "num_edges",
                "original_support",
                "pattern"
            ],
            ascending=[
                True,
                False,
                False,
                False,
                False,
                True
            ]
        )
        .reset_index(drop=True)
    )


# ==========================================================
# SUMMARY BY CATEGORY AND SUPPORT RATIO
# ==========================================================

df_bootstrap_iteration_metrics = pd.DataFrame(
    iteration_metric_rows
)

if not df_bootstrap_iteration_metrics.empty:

    df_bootstrap_group_summary = (
        df_bootstrap_iteration_metrics
        .groupby(
            [
                "category",
                "support_ratio"
            ],
            as_index=False
        )
        .agg(
            successful_iterations=(
                "iteration",
                "nunique"
            ),
            mean_original_pattern_recall_pct=(
                "original_pattern_recall_pct",
                "mean"
            ),
            median_original_pattern_recall_pct=(
                "original_pattern_recall_pct",
                "median"
            ),
            mean_bootstrap_pattern_precision_pct=(
                "bootstrap_pattern_precision_pct",
                "mean"
            ),
            mean_jaccard_similarity_pct=(
                "jaccard_similarity_pct",
                "mean"
            ),
            exact_same_pattern_set_rate_pct=(
                "exact_same_pattern_set",
                lambda values: 100 * values.mean()
            )
        )
        .sort_values(
            [
                "category",
                "support_ratio"
            ]
        )
        .reset_index(drop=True)
    )

else:
    df_bootstrap_group_summary = pd.DataFrame()


df_bootstrap_execution_errors = pd.DataFrame(
    execution_error_rows
)


# ==========================================================
# SAVE RESULTS
# ==========================================================

bootstrap_result_files = {
    "pattern_stability": os.path.join(
        results_directory,
        "bootstrap_pattern_stability.csv"
    ),
    "group_summary": os.path.join(
        results_directory,
        "bootstrap_group_summary.csv"
    ),
    "iteration_metrics": os.path.join(
        results_directory,
        "bootstrap_iteration_metrics.csv"
    ),
    "category_selection": os.path.join(
        results_directory,
        "bootstrap_category_selection.csv"
    ),
    "execution_errors": os.path.join(
        results_directory,
        "bootstrap_execution_errors.csv"
    ),
    "baseline_summary": os.path.join(
        results_directory,
        "bootstrap_baseline_summary.csv"
    )
}

df_bootstrap_pattern_stability.to_csv(
    bootstrap_result_files[
        "pattern_stability"
    ],
    index=False,
    encoding="utf-8-sig"
)

df_bootstrap_group_summary.to_csv(
    bootstrap_result_files[
        "group_summary"
    ],
    index=False,
    encoding="utf-8-sig"
)

df_bootstrap_iteration_metrics.to_csv(
    bootstrap_result_files[
        "iteration_metrics"
    ],
    index=False,
    encoding="utf-8-sig"
)

df_bootstrap_category_selection.to_csv(
    bootstrap_result_files[
        "category_selection"
    ],
    index=False,
    encoding="utf-8-sig"
)

df_bootstrap_execution_errors.to_csv(
    bootstrap_result_files[
        "execution_errors"
    ],
    index=False,
    encoding="utf-8-sig"
)

df_bootstrap_baseline_summary.to_csv(
    bootstrap_result_files[
        "baseline_summary"
    ],
    index=False,
    encoding="utf-8-sig"
)


# ==========================================================
# DISPLAY RESULTS
# ==========================================================

print("\nBootstrap group summary:")

display(
    df_bootstrap_group_summary
)

print(
    "\nPattern stability results "
    "(first 200 rows):"
)

if df_bootstrap_pattern_stability.empty:

    print(
        "No original patterns were available for validation "
        "with the current configuration."
    )

else:

    display(
        df_bootstrap_pattern_stability[
            [
                "category",
                "support_ratio",
                "pattern_key",
                "pattern",
                "num_nodes",
                "num_edges",
                "original_support",
                "bootstrap_reproductions",
                "successful_bootstrap_iterations",
                "bootstrap_frequency_pct",
                "bootstrap_frequency_ci95_low_pct",
                "bootstrap_frequency_ci95_high_pct",
                "stability_class"
            ]
        ].head(200)
    )

if df_bootstrap_execution_errors.empty:

    print(
        "\nBootstrap validation completed without execution errors."
    )

else:

    print(
        "\nWARNING: some bootstrap runs failed. "
        "Review the following table before interpreting results."
    )

    display(
        df_bootstrap_execution_errors.head(20)
    )

print("\nSaved output files:")

for file_label, file_path in (
    bootstrap_result_files.items()
):
    print(
        f"- {file_label}: {file_path}"
    )

Bootstrap category selection:


,category,available_graphs,original_gspan_graphs,selected_for_bootstrap,selection_reason
0,Edge,106,106,True,Eligible
1,FaaS,224,224,True,Eligible
2,HPC,15,15,False,Explicitly excluded; Fewer than 70 graphs
3,None,40,40,False,Explicitly excluded; Fewer than 70 graphs
4,Serverless,352,352,True,Eligible
5,Strictly Edge,77,77,True,Eligible


Original baseline patterns to validate:


,category,support_ratio,baseline_patterns,baseline_structural_patterns
0,Edge,0.03,96,28
1,Edge,0.06,24,3
2,Edge,0.10,10,0
3,FaaS,0.03,92,23
4,FaaS,0.06,29,5
5,FaaS,0.10,8,1
6,Serverless,0.03,77,9
7,Serverless,0.06,12,1
8,Serverless,0.10,6,0
9,Strictly Edge,0.03,101,29



Bootstrap validation for Edge: 106 graphs, 10 iterations
  Completed 1/10 iterations
  Completed 2/10 iterations
  Completed 3/10 iterations
  Completed 4/10 iterations
  Completed 5/10 iterations
  Completed 6/10 iterations
  Completed 7/10 iterations
  Completed 8/10 iterations
  Completed 9/10 iterations
  Completed 10/10 iterations

Bootstrap validation for Serverless: 352 graphs, 10 iterations
  Completed 1/10 iterations
  Completed 2/10 iterations
  Completed 3/10 iterations
  Completed 4/10 iterations
  Completed 5/10 iterations
  Completed 6/10 iterations
  Completed 7/10 iterations
  Completed 8/10 iterations
  Completed 9/10 iterations
  Completed 10/10 iterations

Bootstrap validation for Strictly Edge: 77 graphs, 10 iterations
  Completed 1/10 iterations
  Completed 2/10 iterations
  Completed 3/10 iterations
  Completed 4/10 iterations
  Completed 5/10 iterations
  Completed 6/10 iterations
  Completed 7/10 iterations
  Completed 8/10 iterations
  Completed 9/10 iteration

,category,support_ratio,successful_iterations,mean_original_pattern_recall_pct,median_original_pattern_recall_pct,mean_bootstrap_pattern_precision_pct,mean_jaccard_similarity_pct,exact_same_pattern_set_rate_pct
0,Edge,0.03,10,75.833333,76.562500,32.571961,29.158809,0.0
1,Edge,0.06,10,86.666667,87.500000,53.500120,49.268467,0.0
2,Edge,0.10,10,78.000000,80.000000,64.929205,54.833764,0.0
3,FaaS,0.03,10,83.804348,84.239130,65.506746,57.887898,0.0
4,FaaS,0.06,10,83.103448,84.482759,74.195669,64.130766,0.0
5,FaaS,0.10,10,90.000000,87.500000,71.518398,65.914918,0.0
6,Serverless,0.03,10,79.090909,76.623377,81.117280,66.816754,0.0
7,Serverless,0.06,10,89.166667,91.666667,80.646562,72.481119,0.0
8,Serverless,0.10,10,78.333333,75.000000,91.833333,72.738095,20.0
9,Strictly Edge,0.03,10,73.861386,74.257426,19.853935,18.356922,0.0



Pattern stability results (first 200 rows):


,category,support_ratio,pattern_key,pattern,num_nodes,num_edges,original_support,bootstrap_reproductions,successful_bootstrap_iterations,bootstrap_frequency_pct,bootstrap_frequency_ci95_low_pct,bootstrap_frequency_ci95_high_pct,stability_class
0,Edge,0.10,f10816c734b8776f,UserConsumer -[data]-> CloudFront,2,1,19,10,10,100.0,72.246720,100.000000,High stability
1,Edge,0.10,ef2ee9fe3775ddfb,CloudFront -[data]-> UserConsumer,2,1,15,10,10,100.0,72.246720,100.000000,High stability
2,Edge,0.10,1eb84966f2b0a7e7,Lambda -[data]-> DynamoDB,2,1,15,9,10,90.0,59.584997,98.212379,Moderate stability
3,Edge,0.10,402bc2ab64cd532b,S3 -[data]-> CloudFront,2,1,15,9,10,90.0,59.584997,98.212379,Moderate stability
4,Edge,0.10,c8232e306dc5c3c4,Firehose -[data]-> S3,2,1,13,9,10,90.0,59.584997,98.212379,Moderate stability
...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,FaaS,0.03,2eb863b36c204568,S3 -[data]-> Athena,2,1,15,10,10,100.0,72.246720,100.000000,High stability
196,FaaS,0.03,324adaeeddc247a2,ThirdParty -[data]-> Lambda,2,1,15,10,10,100.0,72.246720,100.000000,High stability
197,FaaS,0.03,7a617fb63af3938e,Lambda -[data]-> RDS,2,1,14,10,10,100.0,72.246720,100.000000,High stability
198,FaaS,0.03,fae5bfbd7839d991,SageMaker -[data]-> Lambda,2,1,14,10,10,100.0,72.246720,100.000000,High stability



Bootstrap validation completed without execution errors.

Saved output files:
- pattern_stability: /content/Cloudscape/gspan_normalized_by_category/bootstrap_validation/results/bootstrap_pattern_stability.csv
- group_summary: /content/Cloudscape/gspan_normalized_by_category/bootstrap_validation/results/bootstrap_group_summary.csv
- iteration_metrics: /content/Cloudscape/gspan_normalized_by_category/bootstrap_validation/results/bootstrap_iteration_metrics.csv
- category_selection: /content/Cloudscape/gspan_normalized_by_category/bootstrap_validation/results/bootstrap_category_selection.csv
- execution_errors: /content/Cloudscape/gspan_normalized_by_category/bootstrap_validation/results/bootstrap_execution_errors.csv
- baseline_summary: /content/Cloudscape/gspan_normalized_by_category/bootstrap_validation/results/bootstrap_baseline_summary.csv


In [ ]:
# ==========================================================
# BOOTSTRAP VALIDATION OF gSpan RESULTS
# ==========================================================

import os
import json
import math
import hashlib
import subprocess

from collections import Counter, defaultdict
from itertools import permutations

import numpy as np
import pandas as pd
import networkx as nx

from IPython.display import display


# ==========================================================
# COMPATIBILITY ALIASES FOR THE PREVIOUS GSPAN CELLS
# These aliases keep all NEW bootstrap code in English while
# using objects that were already created in previous cells.
# ==========================================================

required_previous_objects = [
    "files_norm",
    "category_architectures_norm",
    "category_gspan_info_norm",
    "global_service_to_id_norm",
    "global_edge_type_to_id_norm",
    "global_id_to_service_norm",
    "global_id_to_edge_type_norm",
    "normalizar_nombre_arquitectura_gspan",
    "normalizar_servicio_gspan",
    "normalizar_tipo_arista_gspan",
    "nombre_seguro_categoria_gspan",
    "parsear_salida_gspan_norm",
    "salida_gspan_valida_norm",
    "SUPPORT_RATIOS_NORM",
    "MIN_NODES_NORM",
    "MAX_NODES_NORM",
    "OUTPUT_DIR_NORM"
]

missing_previous_objects = [
    object_name
    for object_name in required_previous_objects
    if object_name not in globals()
]

if missing_previous_objects:
    raise RuntimeError(
        "Run all original gSpan cells before running this "
        "bootstrap validation cell.\n\n"
        f"Missing objects:\n{missing_previous_objects}"
    )

# English aliases for objects created in the original cells.
graph_files = files_norm
category_architectures = category_architectures_norm
category_gspan_info = category_gspan_info_norm

service_to_id = global_service_to_id_norm
edge_type_to_id = global_edge_type_to_id_norm

id_to_service = global_id_to_service_norm
id_to_edge_type = global_id_to_edge_type_norm

normalize_architecture_name = (
    normalizar_nombre_arquitectura_gspan
)

normalize_service = normalizar_servicio_gspan
normalize_edge_type = normalizar_tipo_arista_gspan

safe_category_name = nombre_seguro_categoria_gspan
parse_gspan_output = parsear_salida_gspan_norm
is_valid_gspan_result = salida_gspan_valida_norm

support_ratios = list(SUPPORT_RATIOS_NORM)
min_nodes = MIN_NODES_NORM
max_nodes = MAX_NODES_NORM
main_output_directory = OUTPUT_DIR_NORM


# ==========================================================
# CONFIGURATION
# ==========================================================

# Start with 10 iterations.
# Change only this value to 5000 after verifying that all
# tables and CSV files are generated correctly.
BOOTSTRAP_ITERATIONS = 1000

# Fixed seed for reproducible bootstrap samples.
BOOTSTRAP_RANDOM_SEED = 20260704

# Do not run bootstrap for groups with fewer than 100 graphs.
BOOTSTRAP_MINIMUM_GRAPHS = 70

# Explicit exclusions requested for small groups.
BOOTSTRAP_EXCLUDED_CATEGORIES = {
    "HPC",
    "None"
}

# Use the same support ratios as the original gSpan analysis.
BOOTSTRAP_SUPPORT_RATIOS = support_ratios.copy()

# patterns stored in df_structural_patterns_norm_v2.
BOOTSTRAP_ONLY_STRUCTURAL_PATTERNS = False

# Do not change this to True for the 5000-iteration run.
# Otherwise, thousands of temporary .data files will remain.
BOOTSTRAP_KEEP_TEMPORARY_INPUTS = False


# ==========================================================
# INPUT VALIDATION
# ==========================================================

if BOOTSTRAP_ITERATIONS < 1:
    raise ValueError(
        "BOOTSTRAP_ITERATIONS must be greater than or equal to 1."
    )

if not BOOTSTRAP_SUPPORT_RATIOS:
    raise ValueError(
        "BOOTSTRAP_SUPPORT_RATIOS cannot be empty."
    )

invalid_support_ratios = [
    support_ratio
    for support_ratio in BOOTSTRAP_SUPPORT_RATIOS
    if support_ratio <= 0 or support_ratio > 1
]

if invalid_support_ratios:
    raise ValueError(
        "Every support ratio must be within the interval (0, 1].\n"
        f"Invalid values: {invalid_support_ratios}"
    )


# ==========================================================
# OUTPUT DIRECTORIES
# ==========================================================

bootstrap_directory = os.path.join(
    main_output_directory,
    "bootstrap_validation"
)

temporary_input_directory = os.path.join(
    bootstrap_directory,
    "temporary_inputs"
)

results_directory = os.path.join(
    bootstrap_directory,
    "results"
)

os.makedirs(
    temporary_input_directory,
    exist_ok=True
)

os.makedirs(
    results_directory,
    exist_ok=True
)


# ==========================================================
# GRAPHML FILE INDEX
# ==========================================================

graph_file_by_architecture = {}

for graph_file in graph_files:

    architecture_name = normalize_architecture_name(
        graph_file
    )

    if architecture_name in graph_file_by_architecture:
        raise ValueError(
            "Duplicate normalized architecture name found "
            "among GraphML files:\n"
            f"{architecture_name}"
        )

    graph_file_by_architecture[
        architecture_name
    ] = graph_file


# ==========================================================
# GRAPH CACHE
# ==========================================================

graph_cache = {}


def get_cached_graph(architecture_name):
    """
    Loads a GraphML graph only once and then keeps it in memory.

    This avoids repeatedly reading the same GraphML file during
    bootstrap iterations.
    """

    if architecture_name not in graph_file_by_architecture:
        raise KeyError(
            "No GraphML file was found for architecture:\n"
            f"{architecture_name}"
        )

    if architecture_name not in graph_cache:
        graph_cache[architecture_name] = nx.read_graphml(
            graph_file_by_architecture[
                architecture_name
            ]
        )

    return graph_cache[architecture_name]


# ==========================================================
# CANONICAL STRUCTURAL SIGNATURE FOR PATTERN COMPARISON
# ==========================================================

def create_canonical_pattern_signature(nodes, edges):
    """
    Creates a canonical structural signature for a gSpan pattern.

    The signature is independent of:
    - gSpan pattern IDs;
    - internal node numbering;
    - pattern discovery order.

    The signature preserves:
    - node labels;
    - directed edges;
    - edge labels;
    - graph structure.

    Since max_nodes is 4, testing all node permutations remains
    computationally manageable.
    """

    original_node_ids = sorted(nodes)

    if not original_node_ids:
        raise ValueError(
            "A pattern without nodes cannot have a signature."
        )

    candidate_signatures = []

    for ordered_node_ids in permutations(
        original_node_ids
    ):

        old_to_new_node_id = {
            old_node_id: new_node_id
            for new_node_id, old_node_id in enumerate(
                ordered_node_ids
            )
        }

        ordered_node_labels = tuple(
            int(
                nodes[old_node_id]["label_id"]
            )
            for old_node_id in ordered_node_ids
        )

        ordered_edges = tuple(
            sorted(
                (
                    old_to_new_node_id[
                        edge["source"]
                    ],
                    old_to_new_node_id[
                        edge["target"]
                    ],
                    int(edge["label_id"])
                )
                for edge in edges
            )
        )

        candidate_signatures.append(
            (
                ordered_node_labels,
                ordered_edges
            )
        )

    best_signature = min(
        candidate_signatures
    )

    return json.dumps(
        {
            "node_labels": list(
                best_signature[0]
            ),
            "edges": [
                list(edge)
                for edge in best_signature[1]
            ]
        },
        separators=(",", ":")
    )


def extract_pattern_signatures(gspan_stdout):
    """
    Extracts t/v/e pattern blocks from gSpan output and returns:

        pattern_id -> canonical structural signature
    """

    signatures = {}

    current_pattern = None

    def save_current_pattern():

        if current_pattern is None:
            return

        signatures[
            current_pattern["pattern_id"]
        ] = create_canonical_pattern_signature(
            current_pattern["nodes"],
            current_pattern["edges"]
        )

    for raw_line in gspan_stdout.splitlines():

        line = raw_line.strip()

        if not line:
            continue

        # Pattern beginning.
        if line.startswith("t #"):

            save_current_pattern()

            pattern_id = int(
                line.split()[-1]
            )

            if pattern_id < 0:
                current_pattern = None

            else:
                current_pattern = {
                    "pattern_id": pattern_id,
                    "nodes": {},
                    "edges": []
                }

        # Pattern node.
        elif (
            line.startswith("v ")
            and current_pattern is not None
        ):

            _, node_id, label_id = line.split()[:3]

            current_pattern["nodes"][
                int(node_id)
            ] = {
                "label_id": int(label_id)
            }

        # Pattern edge.
        elif (
            line.startswith("e ")
            and current_pattern is not None
        ):

            (
                _,
                source_node,
                target_node,
                edge_label_id
            ) = line.split()[:4]

            current_pattern["edges"].append({
                "source": int(source_node),
                "target": int(target_node),
                "label_id": int(edge_label_id)
            })

    save_current_pattern()

    return signatures


def create_pattern_key(pattern_signature):
    """
    Creates a short readable identifier for a structural signature.
    """

    return hashlib.sha1(
        pattern_signature.encode("utf-8")
    ).hexdigest()[:16]


# ==========================================================
# BOOTSTRAP GSPAN INPUT FILE CREATION
# ==========================================================

def write_bootstrap_gspan_input(
    category_name,
    sampled_architectures,
    iteration
):
    """
    Writes one bootstrap sample as a gSpan input file.

    IMPORTANT:
    sampled_architectures must be a list-like sequence, not a set.

    Bootstrap uses sampling with replacement. Therefore, the same
    architecture may appear multiple times. Each occurrence is written
    as an independent graph with a different graph ID.
    """

    if not isinstance(
        sampled_architectures,
        (
            list,
            tuple,
            np.ndarray
        )
    ):
        raise TypeError(
            "sampled_architectures must be a list, tuple, or array. "
            "Do not use a set because bootstrap duplicates must remain."
        )

    if len(sampled_architectures) == 0:
        raise ValueError(
            "The bootstrap sample cannot be empty."
        )

    category_slug = safe_category_name(
        category_name
    )

    output_file = os.path.join(
        temporary_input_directory,
        f"{category_slug}_"
        f"bootstrap_{iteration:05d}.data"
    )

    graph_id_to_architecture = {}

    with open(
        output_file,
        "w",
        encoding="utf-8"
    ) as output_file_handle:

        for graph_id, architecture_name in enumerate(
            sampled_architectures
        ):

            graph = get_cached_graph(
                architecture_name
            )

            # The same architecture can appear multiple times.
            # Each appearance receives a distinct graph ID.
            graph_id_to_architecture[
                graph_id
            ] = architecture_name

            output_file_handle.write(
                f"t # {graph_id}\n"
            )

            local_node_ids = {
                node: local_node_id
                for local_node_id, node in enumerate(
                    graph.nodes()
                )
            }

            # Write nodes.
            for node, attributes in graph.nodes(data=True):

                normalized_service = normalize_service(
                    attributes.get(
                        "service",
                        ""
                    )
                )

                service_id = service_to_id[
                    normalized_service
                ]

                output_file_handle.write(
                    f"v "
                    f"{local_node_ids[node]} "
                    f"{service_id}\n"
                )

            # Write directed edges.
            for (
                source_node,
                target_node,
                attributes
            ) in graph.edges(data=True):

                normalized_edge_type = normalize_edge_type(
                    attributes.get(
                        "type",
                        ""
                    )
                )

                edge_type_id = edge_type_to_id[
                    normalized_edge_type
                ]

                output_file_handle.write(
                    f"e "
                    f"{local_node_ids[source_node]} "
                    f"{local_node_ids[target_node]} "
                    f"{edge_type_id}\n"
                )

        output_file_handle.write(
            "t # -1\n"
        )

    return {
        "gspan_file": output_file,
        "graph_id_to_architecture": (
            graph_id_to_architecture
        ),
        "number_of_graphs": len(
            sampled_architectures
        )
    }


# ==========================================================
# RUN GSPAN AND ADD STRUCTURAL PATTERN SIGNATURES
# ==========================================================

def run_gspan_for_bootstrap(
    gspan_file,
    graph_id_to_architecture,
    number_of_graphs,
    support_ratio
):
    """
    Runs gSpan and returns a DataFrame with:

    - pattern_signature
    - pattern_key

    Returns:
        patterns_dataframe, None

    In case of a real execution error:
        empty_dataframe, error_message
    """

    minimum_support = max(
        2,
        math.ceil(
            number_of_graphs
            * support_ratio
        )
    )

    command = [
        "python",
        "-m",
        "gspan_mining",
        "-s",
        str(minimum_support),
        "-l",
        str(min_nodes),
        "-u",
        str(max_nodes),
        "-d",
        "True",
        "-w",
        "True",
        gspan_file
    ]

    result = subprocess.run(
        command,
        capture_output=True,
        text=True
    )

    if not is_valid_gspan_result(result):

        stdout_preview = "\n".join(
            result.stdout
            .splitlines()[:30]
        )

        error_message = (
            f"returncode={result.returncode}\n\n"
            f"stderr:\n{result.stderr[:3000]}\n\n"
            f"stdout preview:\n{stdout_preview}"
        )

        return pd.DataFrame(), error_message

    patterns_dataframe = parse_gspan_output(
        texto=result.stdout,
        catalogo_nodos=id_to_service,
        catalogo_aristas=id_to_edge_type,
        catalogo_grafos=graph_id_to_architecture,
        total_grafos=number_of_graphs
    )

    if patterns_dataframe.empty:
        return patterns_dataframe, None

    pattern_signatures = extract_pattern_signatures(
        result.stdout
    )

    patterns_dataframe = patterns_dataframe.copy()

    patterns_dataframe["pattern_signature"] = (
        patterns_dataframe["pattern_id"].map(
            pattern_signatures
        )
    )

    unresolved_pattern_ids = (
        patterns_dataframe.loc[
            patterns_dataframe[
                "pattern_signature"
            ].isna(),
            "pattern_id"
        ]
        .tolist()
    )

    if unresolved_pattern_ids:
        return (
            pd.DataFrame(),
            (
                "A canonical signature could not be generated for "
                "these pattern IDs:\n"
                f"{unresolved_pattern_ids}"
            )
        )

    patterns_dataframe["pattern_key"] = (
        patterns_dataframe[
            "pattern_signature"
        ].map(
            create_pattern_key
        )
    )

    # A structural signature should identify each graph pattern
    # uniquely. This removes any accidental duplicates safely.
    patterns_dataframe = (
        patterns_dataframe
        .drop_duplicates(
            subset=["pattern_signature"]
        )
        .reset_index(drop=True)
    )

    return patterns_dataframe, None


# ==========================================================
# BOOTSTRAP METRIC FUNCTIONS
# ==========================================================

def wilson_confidence_interval(
    successes,
    total,
    z_score=1.959963984540054
):
    """
    Calculates a 95% Wilson confidence interval for a proportion.
    """

    if total <= 0:
        return np.nan, np.nan

    proportion = successes / total
    z_squared = z_score ** 2

    denominator = (
        1
        + z_squared / total
    )

    center = (
        proportion
        + z_squared / (2 * total)
    ) / denominator

    half_width = (
        z_score
        * math.sqrt(
            (
                proportion
                * (1 - proportion)
                + z_squared / (4 * total)
            )
            / total
        )
        / denominator
    )

    lower_bound = max(
        0,
        center - half_width
    )

    upper_bound = min(
        1,
        center + half_width
    )

    return lower_bound, upper_bound


def classify_stability(bootstrap_frequency_percentage):
    """
    Assigns an interpretable stability class.
    """

    if pd.isna(
        bootstrap_frequency_percentage
    ):
        return "Not estimated"

    if bootstrap_frequency_percentage >= 95:
        return "High stability"

    if bootstrap_frequency_percentage >= 80:
        return "Moderate stability"

    return "Low stability"


# ==========================================================
# SELECT ELIGIBLE CATEGORIES
# ==========================================================

bootstrap_population_by_category = {}

category_selection_rows = []

for (
    category_name,
    category_architecture_names
) in category_architectures.items():

    available_architectures = sorted(
        set(category_architecture_names)
        & set(graph_file_by_architecture)
    )

    original_gspan_graph_count = (
        category_gspan_info[
            category_name
        ]["num_graphs"]
    )

    exclusion_reasons = []

    if category_name in BOOTSTRAP_EXCLUDED_CATEGORIES:
        exclusion_reasons.append(
            "Explicitly excluded"
        )

    if len(available_architectures) < BOOTSTRAP_MINIMUM_GRAPHS:
        exclusion_reasons.append(
            f"Fewer than "
            f"{BOOTSTRAP_MINIMUM_GRAPHS} graphs"
        )

    if len(available_architectures) != original_gspan_graph_count:
        exclusion_reasons.append(
            "Graph count differs from the original gSpan input"
        )

    is_selected = not exclusion_reasons

    category_selection_rows.append({
        "category": category_name,
        "available_graphs": len(
            available_architectures
        ),
        "original_gspan_graphs": (
            original_gspan_graph_count
        ),
        "selected_for_bootstrap": is_selected,
        "selection_reason": (
            "Eligible"
            if is_selected
            else "; ".join(
                exclusion_reasons
            )
        )
    })

    if is_selected:
        bootstrap_population_by_category[
            category_name
        ] = available_architectures


df_bootstrap_category_selection = (
    pd.DataFrame(
        category_selection_rows
    )
    .sort_values(
        "category"
    )
    .reset_index(drop=True)
)

print(
    "Bootstrap category selection:"
)

display(
    df_bootstrap_category_selection
)

if not bootstrap_population_by_category:
    raise RuntimeError(
        "No category is eligible for bootstrap validation with "
        "the current configuration."
    )


# ==========================================================
# RUN ORIGINAL BASELINE PATTERNS
# ==========================================================

baseline_patterns = {}

baseline_error_rows = []

for category_name in bootstrap_population_by_category:

    category_information = category_gspan_info[
        category_name
    ]

    for support_ratio in BOOTSTRAP_SUPPORT_RATIOS:

        (
            baseline_dataframe,
            baseline_error
        ) = run_gspan_for_bootstrap(
            gspan_file=category_information[
                "gspan_file"
            ],
            graph_id_to_architecture=category_information[
                "graph_id_to_name"
            ],
            number_of_graphs=category_information[
                "num_graphs"
            ],
            support_ratio=support_ratio
        )

        if baseline_error is not None:

            baseline_error_rows.append({
                "category": category_name,
                "support_ratio": support_ratio,
                "error": baseline_error
            })

            continue

        if (
            BOOTSTRAP_ONLY_STRUCTURAL_PATTERNS
            and not baseline_dataframe.empty
        ):
            baseline_dataframe = (
                baseline_dataframe.loc[
                    baseline_dataframe[
                        "num_edges"
                    ] >= 2
                ]
                .copy()
            )

        baseline_patterns[
            (
                category_name,
                support_ratio
            )
        ] = baseline_dataframe


if baseline_error_rows:

    print(
        "Errors occurred while generating the original baseline "
        "patterns. Bootstrap validation was not started."
    )

    display(
        pd.DataFrame(
            baseline_error_rows
        )
    )

    raise RuntimeError(
        "Baseline gSpan execution failed. Resolve the errors before "
        "running bootstrap validation."
    )


baseline_summary_rows = []

for (
    category_name,
    support_ratio
), baseline_dataframe in baseline_patterns.items():

    total_baseline_patterns = len(
        baseline_dataframe
    )

    structural_baseline_patterns = int(
        (
            baseline_dataframe[
                "num_edges"
            ] >= 2
        ).sum()
    ) if not baseline_dataframe.empty else 0

    baseline_summary_rows.append({
        "category": category_name,
        "support_ratio": support_ratio,
        "baseline_patterns": (
            total_baseline_patterns
        ),
        "baseline_structural_patterns": (
            structural_baseline_patterns
        )
    })


df_bootstrap_baseline_summary = (
    pd.DataFrame(
        baseline_summary_rows
    )
    .sort_values(
        [
            "category",
            "support_ratio"
        ]
    )
    .reset_index(drop=True)
)

print(
    "Original baseline patterns to validate:"
)

display(
    df_bootstrap_baseline_summary
)


# ==========================================================
# EXECUTE BOOTSTRAP ITERATIONS
# ==========================================================

random_generator = np.random.default_rng(
    BOOTSTRAP_RANDOM_SEED
)

# Number of bootstrap reproductions for every original pattern.
pattern_reproductions = Counter()

# Bootstrap support values observed when original patterns reappear.
recovered_support_values = defaultdict(list)

# Number of successful executions by category and support ratio.
successful_runs = Counter()

# General metrics for each bootstrap iteration.
iteration_metric_rows = []

# Real gSpan execution errors.
execution_error_rows = []


for (
    category_name,
    category_population
) in bootstrap_population_by_category.items():

    category_size = len(
        category_population
    )

    print(
        f"\nBootstrap validation for {category_name}: "
        f"{category_size} graphs, "
        f"{BOOTSTRAP_ITERATIONS} iterations"
    )

    progress_step = max(
        1,
        BOOTSTRAP_ITERATIONS // 10
    )

    for iteration in range(
        1,
        BOOTSTRAP_ITERATIONS + 1
    ):

        # Sample the same number of graphs with replacement.
        sampled_architectures = (
            random_generator.choice(
                category_population,
                size=category_size,
                replace=True
            )
            .tolist()
        )

        bootstrap_input_information = None

        try:

            bootstrap_input_information = (
                write_bootstrap_gspan_input(
                    category_name=category_name,
                    sampled_architectures=sampled_architectures,
                    iteration=iteration
                )
            )

            # Reuse the exact same sample for all support thresholds.
            for support_ratio in BOOTSTRAP_SUPPORT_RATIOS:

                baseline_dataframe = baseline_patterns[
                    (
                        category_name,
                        support_ratio
                    )
                ]

                (
                    bootstrap_dataframe,
                    execution_error
                ) = run_gspan_for_bootstrap(
                    gspan_file=bootstrap_input_information[
                        "gspan_file"
                    ],
                    graph_id_to_architecture=(
                        bootstrap_input_information[
                            "graph_id_to_architecture"
                        ]
                    ),
                    number_of_graphs=(
                        bootstrap_input_information[
                            "number_of_graphs"
                        ]
                    ),
                    support_ratio=support_ratio
                )

                if execution_error is not None:

                    execution_error_rows.append({
                        "category": category_name,
                        "iteration": iteration,
                        "support_ratio": support_ratio,
                        "error": execution_error
                    })

                    continue

                if (
                    BOOTSTRAP_ONLY_STRUCTURAL_PATTERNS
                    and not bootstrap_dataframe.empty
                ):
                    bootstrap_dataframe = (
                        bootstrap_dataframe.loc[
                            bootstrap_dataframe[
                                "num_edges"
                            ] >= 2
                        ]
                        .copy()
                    )

                successful_runs[
                    (
                        category_name,
                        support_ratio
                    )
                ] += 1

                # When baseline has no patterns, there are no
                # individual patterns to validate.
                if baseline_dataframe.empty:

                    iteration_metric_rows.append({
                        "category": category_name,
                        "iteration": iteration,
                        "support_ratio": support_ratio,
                        "original_pattern_count": 0,
                        "bootstrap_pattern_count": len(
                            bootstrap_dataframe
                        ),
                        "shared_pattern_count": 0,
                        "original_pattern_recall_pct": np.nan,
                        "bootstrap_pattern_precision_pct": np.nan,
                        "jaccard_similarity_pct": np.nan,
                        "exact_same_pattern_set": (
                            bootstrap_dataframe.empty
                        )
                    })

                    continue

                original_signatures = set(
                    baseline_dataframe[
                        "pattern_signature"
                    ]
                )

                bootstrap_signatures = set(
                    bootstrap_dataframe[
                        "pattern_signature"
                    ]
                ) if not bootstrap_dataframe.empty else set()

                shared_signatures = (
                    original_signatures
                    & bootstrap_signatures
                )

                union_signatures = (
                    original_signatures
                    | bootstrap_signatures
                )

                if bootstrap_dataframe.empty:
                    support_by_signature = {}

                else:
                    support_by_signature = (
                        bootstrap_dataframe
                        .groupby(
                            "pattern_signature"
                        )["support"]
                        .max()
                        .to_dict()
                    )

                # Count one reproduction for each original pattern
                # that appears again in the bootstrap sample.
                for pattern_signature in shared_signatures:

                    reproduction_key = (
                        category_name,
                        support_ratio,
                        pattern_signature
                    )

                    pattern_reproductions[
                        reproduction_key
                    ] += 1

                    recovered_support_values[
                        reproduction_key
                    ].append(
                        support_by_signature[
                            pattern_signature
                        ]
                    )

                original_pattern_count = len(
                    original_signatures
                )

                bootstrap_pattern_count = len(
                    bootstrap_signatures
                )

                shared_pattern_count = len(
                    shared_signatures
                )

                original_pattern_recall = (
                    shared_pattern_count
                    / original_pattern_count
                    if original_pattern_count > 0
                    else np.nan
                )

                bootstrap_pattern_precision = (
                    shared_pattern_count
                    / bootstrap_pattern_count
                    if bootstrap_pattern_count > 0
                    else np.nan
                )

                jaccard_similarity = (
                    shared_pattern_count
                    / len(union_signatures)
                    if union_signatures
                    else np.nan
                )

                iteration_metric_rows.append({
                    "category": category_name,
                    "iteration": iteration,
                    "support_ratio": support_ratio,
                    "original_pattern_count": (
                        original_pattern_count
                    ),
                    "bootstrap_pattern_count": (
                        bootstrap_pattern_count
                    ),
                    "shared_pattern_count": (
                        shared_pattern_count
                    ),
                    "original_pattern_recall_pct": (
                        100
                        * original_pattern_recall
                    ),
                    "bootstrap_pattern_precision_pct": (
                        100
                        * bootstrap_pattern_precision
                    ),
                    "jaccard_similarity_pct": (
                        100
                        * jaccard_similarity
                    ),
                    "exact_same_pattern_set": (
                        original_signatures
                        == bootstrap_signatures
                    )
                })

        finally:

            if (
                bootstrap_input_information is not None
                and not BOOTSTRAP_KEEP_TEMPORARY_INPUTS
            ):

                temporary_file = (
                    bootstrap_input_information[
                        "gspan_file"
                    ]
                )

                if os.path.exists(temporary_file):
                    os.remove(temporary_file)

        if (
            iteration == 1
            or iteration % progress_step == 0
            or iteration == BOOTSTRAP_ITERATIONS
        ):
            print(
                f"  Completed "
                f"{iteration}/{BOOTSTRAP_ITERATIONS} iterations"
            )


# ==========================================================
# PATTERN STABILITY RESULTS
# ==========================================================

pattern_stability_rows = []

for (
    category_name,
    support_ratio
), baseline_dataframe in baseline_patterns.items():

    successful_iterations = successful_runs[
        (
            category_name,
            support_ratio
        )
    ]

    if baseline_dataframe.empty:
        continue

    for _, original_pattern_row in (
        baseline_dataframe.iterrows()
    ):

        pattern_signature = original_pattern_row[
            "pattern_signature"
        ]

        reproduction_key = (
            category_name,
            support_ratio,
            pattern_signature
        )

        bootstrap_reproductions = pattern_reproductions[
            reproduction_key
        ]

        bootstrap_frequency = (
            bootstrap_reproductions
            / successful_iterations
            if successful_iterations > 0
            else np.nan
        )

        (
            confidence_interval_low,
            confidence_interval_high
        ) = wilson_confidence_interval(
            bootstrap_reproductions,
            successful_iterations
        )

        recovered_supports = recovered_support_values[
            reproduction_key
        ]

        category_graph_count = len(
            bootstrap_population_by_category[
                category_name
            ]
        )

        minimum_support = max(
            2,
            math.ceil(
                category_graph_count
                * support_ratio
            )
        )

        pattern_stability_rows.append({
            "category": category_name,
            "support_ratio": support_ratio,
            "minimum_support": minimum_support,
            "category_graphs": category_graph_count,
            "pattern_key": original_pattern_row[
                "pattern_key"
            ],
            "pattern": original_pattern_row[
                "pattern"
            ],
            "num_nodes": int(
                original_pattern_row[
                    "num_nodes"
                ]
            ),
            "num_edges": int(
                original_pattern_row[
                    "num_edges"
                ]
            ),
            "original_support": int(
                original_pattern_row[
                    "support"
                ]
            ),
            "original_percentage_graphs": float(
                original_pattern_row[
                    "percentage_graphs"
                ]
            ),
            "bootstrap_reproductions": (
                bootstrap_reproductions
            ),
            "successful_bootstrap_iterations": (
                successful_iterations
            ),
            "bootstrap_frequency_pct": (
                100
                * bootstrap_frequency
            ),
            "bootstrap_frequency_ci95_low_pct": (
                100
                * confidence_interval_low
            ),
            "bootstrap_frequency_ci95_high_pct": (
                100
                * confidence_interval_high
            ),
            "mean_support_when_recovered": (
                float(
                    np.mean(
                        recovered_supports
                    )
                )
                if recovered_supports
                else np.nan
            ),
            "stability_class": classify_stability(
                100
                * bootstrap_frequency
            )
        })


df_bootstrap_pattern_stability = pd.DataFrame(
    pattern_stability_rows
)

if not df_bootstrap_pattern_stability.empty:

    df_bootstrap_pattern_stability = (
        df_bootstrap_pattern_stability
        .sort_values(
            [
                "category",
                "support_ratio",
                "bootstrap_frequency_pct",
                "num_edges",
                "original_support",
                "pattern"
            ],
            ascending=[
                True,
                False,
                False,
                False,
                False,
                True
            ]
        )
        .reset_index(drop=True)
    )


# ==========================================================
# SUMMARY BY CATEGORY AND SUPPORT RATIO
# ==========================================================

df_bootstrap_iteration_metrics = pd.DataFrame(
    iteration_metric_rows
)

if not df_bootstrap_iteration_metrics.empty:

    df_bootstrap_group_summary = (
        df_bootstrap_iteration_metrics
        .groupby(
            [
                "category",
                "support_ratio"
            ],
            as_index=False
        )
        .agg(
            successful_iterations=(
                "iteration",
                "nunique"
            ),
            mean_original_pattern_recall_pct=(
                "original_pattern_recall_pct",
                "mean"
            ),
            median_original_pattern_recall_pct=(
                "original_pattern_recall_pct",
                "median"
            ),
            mean_bootstrap_pattern_precision_pct=(
                "bootstrap_pattern_precision_pct",
                "mean"
            ),
            mean_jaccard_similarity_pct=(
                "jaccard_similarity_pct",
                "mean"
            ),
            exact_same_pattern_set_rate_pct=(
                "exact_same_pattern_set",
                lambda values: 100 * values.mean()
            )
        )
        .sort_values(
            [
                "category",
                "support_ratio"
            ]
        )
        .reset_index(drop=True)
    )

else:
    df_bootstrap_group_summary = pd.DataFrame()


df_bootstrap_execution_errors = pd.DataFrame(
    execution_error_rows
)


# ==========================================================
# SAVE RESULTS
# ==========================================================

bootstrap_result_files = {
    "pattern_stability": os.path.join(
        results_directory,
        "bootstrap_pattern_stability.csv"
    ),
    "group_summary": os.path.join(
        results_directory,
        "bootstrap_group_summary.csv"
    ),
    "iteration_metrics": os.path.join(
        results_directory,
        "bootstrap_iteration_metrics.csv"
    ),
    "category_selection": os.path.join(
        results_directory,
        "bootstrap_category_selection.csv"
    ),
    "execution_errors": os.path.join(
        results_directory,
        "bootstrap_execution_errors.csv"
    ),
    "baseline_summary": os.path.join(
        results_directory,
        "bootstrap_baseline_summary.csv"
    )
}

df_bootstrap_pattern_stability.to_csv(
    bootstrap_result_files[
        "pattern_stability"
    ],
    index=False,
    encoding="utf-8-sig"
)

df_bootstrap_group_summary.to_csv(
    bootstrap_result_files[
        "group_summary"
    ],
    index=False,
    encoding="utf-8-sig"
)

df_bootstrap_iteration_metrics.to_csv(
    bootstrap_result_files[
        "iteration_metrics"
    ],
    index=False,
    encoding="utf-8-sig"
)

df_bootstrap_category_selection.to_csv(
    bootstrap_result_files[
        "category_selection"
    ],
    index=False,
    encoding="utf-8-sig"
)

df_bootstrap_execution_errors.to_csv(
    bootstrap_result_files[
        "execution_errors"
    ],
    index=False,
    encoding="utf-8-sig"
)

df_bootstrap_baseline_summary.to_csv(
    bootstrap_result_files[
        "baseline_summary"
    ],
    index=False,
    encoding="utf-8-sig"
)


# ==========================================================
# DISPLAY RESULTS
# ==========================================================

print("\nBootstrap group summary:")

display(
    df_bootstrap_group_summary
)

print(
    "\nPattern stability results "
    "(first 200 rows):"
)

if df_bootstrap_pattern_stability.empty:

    print(
        "No original patterns were available for validation "
        "with the current configuration."
    )

else:

    display(
        df_bootstrap_pattern_stability[
            [
                "category",
                "support_ratio",
                "pattern_key",
                "pattern",
                "num_nodes",
                "num_edges",
                "original_support",
                "bootstrap_reproductions",
                "successful_bootstrap_iterations",
                "bootstrap_frequency_pct",
                "bootstrap_frequency_ci95_low_pct",
                "bootstrap_frequency_ci95_high_pct",
                "stability_class"
            ]
        ].head(200)
    )

if df_bootstrap_execution_errors.empty:

    print(
        "\nBootstrap validation completed without execution errors."
    )

else:

    print(
        "\nWARNING: some bootstrap runs failed. "
        "Review the following table before interpreting results."
    )

    display(
        df_bootstrap_execution_errors.head(20)
    )

print("\nSaved output files:")

for file_label, file_path in (
    bootstrap_result_files.items()
):
    print(
        f"- {file_label}: {file_path}"
    )

Bootstrap category selection:


,category,available_graphs,original_gspan_graphs,selected_for_bootstrap,selection_reason
0,Edge,106,106,True,Eligible
1,FaaS,224,224,True,Eligible
2,HPC,15,15,False,Explicitly excluded; Fewer than 70 graphs
3,None,40,40,False,Explicitly excluded; Fewer than 70 graphs
4,Serverless,352,352,True,Eligible
5,Strictly Edge,77,77,True,Eligible


Original baseline patterns to validate:


,category,support_ratio,baseline_patterns,baseline_structural_patterns
0,Edge,0.03,96,28
1,Edge,0.06,24,3
2,Edge,0.10,10,0
3,FaaS,0.03,92,23
4,FaaS,0.06,29,5
5,FaaS,0.10,8,1
6,Serverless,0.03,77,9
7,Serverless,0.06,12,1
8,Serverless,0.10,6,0
9,Strictly Edge,0.03,101,29



Bootstrap validation for Edge: 106 graphs, 1000 iterations
  Completed 1/1000 iterations
  Completed 100/1000 iterations
  Completed 200/1000 iterations
  Completed 300/1000 iterations
  Completed 400/1000 iterations
  Completed 500/1000 iterations
  Completed 600/1000 iterations
  Completed 700/1000 iterations
  Completed 800/1000 iterations
  Completed 900/1000 iterations
  Completed 1000/1000 iterations

Bootstrap validation for Serverless: 352 graphs, 1000 iterations
  Completed 1/1000 iterations
  Completed 100/1000 iterations
  Completed 200/1000 iterations
  Completed 300/1000 iterations
  Completed 400/1000 iterations
  Completed 500/1000 iterations
  Completed 600/1000 iterations
  Completed 700/1000 iterations
  Completed 800/1000 iterations
  Completed 900/1000 iterations
  Completed 1000/1000 iterations

Bootstrap validation for Strictly Edge: 77 graphs, 1000 iterations
  Completed 1/1000 iterations
  Completed 100/1000 iterations
  Completed 200/1000 iterations
  Complete

,category,support_ratio,successful_iterations,mean_original_pattern_recall_pct,median_original_pattern_recall_pct,mean_bootstrap_pattern_precision_pct,mean_jaccard_similarity_pct,exact_same_pattern_set_rate_pct
0,Edge,0.03,1000,73.248958,73.958333,38.261038,33.120219,0.0
1,Edge,0.06,1000,84.037500,83.333333,57.862801,51.920071,0.0
2,Edge,0.10,1000,77.810000,80.000000,66.643589,55.745219,0.1
3,FaaS,0.03,1000,83.693478,83.695652,68.660192,60.486588,0.0
4,FaaS,0.06,1000,81.320690,82.758621,75.546990,64.317773,0.0
5,FaaS,0.10,1000,89.075000,87.500000,83.358424,75.480553,8.2
6,Serverless,0.03,1000,82.162338,81.818182,78.345250,66.918177,0.0
7,Serverless,0.06,1000,88.841667,91.666667,72.984251,66.755779,0.0
8,Serverless,0.10,1000,81.200000,83.333333,91.367738,75.385913,14.1
9,Strictly Edge,0.03,1000,76.015842,76.237624,21.567195,20.052466,0.0



Pattern stability results (first 200 rows):


,category,support_ratio,pattern_key,pattern,num_nodes,num_edges,original_support,bootstrap_reproductions,successful_bootstrap_iterations,bootstrap_frequency_pct,bootstrap_frequency_ci95_low_pct,bootstrap_frequency_ci95_high_pct,stability_class
0,Edge,0.10,f10816c734b8776f,UserConsumer -[data]-> CloudFront,2,1,19,988,1000,98.8,97.914273,99.312235,High stability
1,Edge,0.10,1eb84966f2b0a7e7,Lambda -[data]-> DynamoDB,2,1,15,909,1000,90.9,88.957442,92.529530,Moderate stability
2,Edge,0.10,ef2ee9fe3775ddfb,CloudFront -[data]-> UserConsumer,2,1,15,902,1000,90.2,88.200530,91.891799,Moderate stability
3,Edge,0.10,402bc2ab64cd532b,S3 -[data]-> CloudFront,2,1,15,897,1000,89.7,87.661636,91.434519,Moderate stability
4,Edge,0.10,c8232e306dc5c3c4,Firehose -[data]-> S3,2,1,13,776,1000,77.6,74.913107,80.075656,Low stability
...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,FaaS,0.03,f10816c734b8776f,UserConsumer -[data]-> CloudFront,2,1,14,986,1000,98.6,97.663797,99.164242,High stability
196,FaaS,0.03,efcc4af655b4dab3,StepFunctions -[data]-> Lambda,2,1,13,982,1000,98.2,97.172674,98.858426,High stability
197,FaaS,0.03,01c0c0488b015fce,Glue -[data]-> S3,2,1,13,980,1000,98.0,96.930999,98.701632,High stability
198,FaaS,0.03,b66ec417df177ee9,Lambda -[data]-> EC2,2,1,13,980,1000,98.0,96.930999,98.701632,High stability



Bootstrap validation completed without execution errors.

Saved output files:
- pattern_stability: /content/Cloudscape/gspan_normalized_by_category/bootstrap_validation/results/bootstrap_pattern_stability.csv
- group_summary: /content/Cloudscape/gspan_normalized_by_category/bootstrap_validation/results/bootstrap_group_summary.csv
- iteration_metrics: /content/Cloudscape/gspan_normalized_by_category/bootstrap_validation/results/bootstrap_iteration_metrics.csv
- category_selection: /content/Cloudscape/gspan_normalized_by_category/bootstrap_validation/results/bootstrap_category_selection.csv
- execution_errors: /content/Cloudscape/gspan_normalized_by_category/bootstrap_validation/results/bootstrap_execution_errors.csv
- baseline_summary: /content/Cloudscape/gspan_normalized_by_category/bootstrap_validation/results/bootstrap_baseline_summary.csv
